In [ ]:
"""Task 3 notebook block 01: runtime identity and source-only guardrails.

This block does not mount Drive, inspect PACS, load a model, or access Sketch.
Run it before any other Task 3 block and preserve the emitted JSON file.
"""

import hashlib
import json
import os
import platform
import random
from pathlib import Path, PurePosixPath

# Required for deterministic CUDA matrix multiplication.
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import numpy as np
import sklearn
import torch
import torchvision


PROTOCOL_VERSION = "task3-approved-2026-09-24-v1"
SEED = 6304
SOURCE_DOMAINS = ("photo", "art_painting", "cartoon")
FORBIDDEN_TARGET_DOMAIN = "sketch"

LOCKED_IDENTITIES = {
    "task2_protocol_sha256": (
        "e0f075e1e4f2c43c7db2423bb9b31f901d4e1157e72c2097b3e156501ce2dc74"
    ),
    "dataset_file_list_sha256": (
        "559ac63b8df8e07330b97112e5ec4c414b3957585b28d21b4cfecd2181f538e0"
    ),
    "common_initialization_state_sha256": (
        "4d53e76c2d8f557b050a1913257c980846bebf6d5b4a28ff4d7cfa12c1d2eef3"
    ),
    "erm_checkpoint_file_sha256": (
        "3d28a223e4b97b323cb3a20dcb5b7577af96631f2e6ef1f2bc99d53d85761327"
    ),
}

LOCKED_SOURCE_COUNTS = {
    "photo": {"train": 1336, "validation": 334},
    "art_painting": {"train": 1638, "validation": 410},
    "cartoon": {"train": 1875, "validation": 469},
}

LOCKED_TRAINING = {
    "seed": 6304,
    "weights": "ResNet18_Weights.IMAGENET1K_V1",
    "num_classes": 7,
    "feature_width": 512,
    "epochs": 30,
    "patience": 5,
    "source_batch_per_domain": 8,
    "steps_per_source_epoch": 235,
    "optimizer": "AdamW",
    "learning_rate": 1e-4,
    "weight_decay": 1e-4,
    "gradient_clipping_max_norm": 20.0,
    "selection_metric": "unweighted_mean_source_validation_macro_f1",
    "dan_dg_main_lambda": 1.0,
    "dan_dg_study_lambdas": [0.1, 1.0, 10.0],
    "sam_rho": 0.05,
    "mmd_feature_normalization": "l2_per_sample_mmd_input_only",
    "adversarial_feature_normalization": "not_applicable_to_task3",
    "mmd_bandwidth_pairs": "strict_upper_triangle_keep_off_diagonal_zeros",
    "mmd_kernel": "exp(-squared_distance/(2*bandwidth))",
    "mmd_kernel_factors": [0.5, 1.0, 2.0],
    "mmd_estimator": "v_statistic_include_within_domain_diagonals",
}


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

    if hasattr(torch.backends.cuda.matmul, "allow_tf32"):
        torch.backends.cuda.matmul.allow_tf32 = False

    if hasattr(torch.backends.cudnn, "allow_tf32"):
        torch.backends.cudnn.allow_tf32 = False

    torch.use_deterministic_algorithms(True)


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()


def require_source_record(record: dict) -> None:
    """Reject any record that is not an approved labeled-source record."""

    if set(record) != {"path", "class_id"}:
        raise ValueError(f"Invalid source-record fields: {sorted(record)}")

    parts = PurePosixPath(str(record["path"])).parts

    if not parts or parts[0] not in SOURCE_DOMAINS:
        raise RuntimeError(
            f"Target or unknown-domain record rejected: {record['path']}"
        )

    if FORBIDDEN_TARGET_DOMAIN in {part.lower() for part in parts}:
        raise RuntimeError(
            f"Sketch record rejected during source-only phase: {record['path']}"
        )

    if not isinstance(record["class_id"], int) or not 0 <= record["class_id"] < 7:
        raise ValueError(f"Invalid class ID in source record: {record}")


seed_everything(SEED)

if not torch.cuda.is_available():
    raise RuntimeError(
        "Task 3 training requires a CUDA runtime. In Colab, select a GPU runtime "
        "and rerun this first block before mounting Drive or loading any data."
    )

runtime_identity = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "numpy": np.__version__,
    "sklearn": sklearn.__version__,
    "cuda": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(),
    "device_type": "cuda",
    "gpu": torch.cuda.get_device_name(0),
}

preflight_record = {
    "status": "TASK3_BLOCK_01_PASS",
    "protocol_version": PROTOCOL_VERSION,
    "source_only_phase": True,
    "drive_mounted_by_this_block": False,
    "dataset_traversed_by_this_block": False,
    "models_loaded_by_this_block": False,
    "sketch_images_accessed": 0,
    "source_domains": list(SOURCE_DOMAINS),
    "forbidden_target_domain": FORBIDDEN_TARGET_DOMAIN,
    "locked_identities": LOCKED_IDENTITIES,
    "locked_source_counts": LOCKED_SOURCE_COUNTS,
    "locked_training": LOCKED_TRAINING,
    "runtime": runtime_identity,
}

output_path = Path("/content/task3_runtime_preflight.json")
output_path.write_text(json.dumps(preflight_record, indent=2) + "\n")

print(json.dumps(preflight_record, indent=2))
print(f"\nSaved: {output_path}")
print("Next action: inspect this record before mounting Drive or supplying any path.")

{
  "status": "TASK3_BLOCK_01_PASS",
  "protocol_version": "task3-approved-2026-09-24-v1",
  "source_only_phase": true,
  "drive_mounted_by_this_block": false,
  "dataset_traversed_by_this_block": false,
  "models_loaded_by_this_block": false,
  "sketch_images_accessed": 0,
  "source_domains": [
    "photo",
    "art_painting",
    "cartoon"
  ],
  "forbidden_target_domain": "sketch",
  "locked_identities": {
    "task2_protocol_sha256": "e0f075e1e4f2c43c7db2423bb9b31f901d4e1157e72c2097b3e156501ce2dc74",
    "dataset_file_list_sha256": "559ac63b8df8e07330b97112e5ec4c414b3957585b28d21b4cfecd2181f538e0",
    "common_initialization_state_sha256": "4d53e76c2d8f557b050a1913257c980846bebf6d5b4a28ff4d7cfa12c1d2eef3",
    "erm_checkpoint_file_sha256": "3d28a223e4b97b323cb3a20dcb5b7577af96631f2e6ef1f2bc99d53d85761327"
  },
  "locked_source_counts": {
    "photo": {
      "train": 1336,
      "validation": 334
    },
    "art_painting": {
      "train": 1638,
      "validation": 410
    },
    "

In [ ]:
from pathlib import Path

preflight_path = Path("/content/task3_runtime_preflight.json")
print("Exists:", preflight_path.exists())
print("Location:", preflight_path.resolve())
print(preflight_path.read_text())

Exists: True
Location: /content/task3_runtime_preflight.json
{
  "status": "TASK3_BLOCK_01_PASS",
  "protocol_version": "task3-approved-2026-09-24-v1",
  "source_only_phase": true,
  "drive_mounted_by_this_block": false,
  "dataset_traversed_by_this_block": false,
  "models_loaded_by_this_block": false,
  "sketch_images_accessed": 0,
  "source_domains": [
    "photo",
    "art_painting",
    "cartoon"
  ],
  "forbidden_target_domain": "sketch",
  "locked_identities": {
    "task2_protocol_sha256": "e0f075e1e4f2c43c7db2423bb9b31f901d4e1157e72c2097b3e156501ce2dc74",
    "dataset_file_list_sha256": "559ac63b8df8e07330b97112e5ec4c414b3957585b28d21b4cfecd2181f538e0",
    "common_initialization_state_sha256": "4d53e76c2d8f557b050a1913257c980846bebf6d5b4a28ff4d7cfa12c1d2eef3",
    "erm_checkpoint_file_sha256": "3d28a223e4b97b323cb3a20dcb5b7577af96631f2e6ef1f2bc99d53d85761327"
  },
  "locked_source_counts": {
    "photo": {
      "train": 1336,
      "validation": 334
    },
    "art_painting"

In [ ]:
"""Task 3 notebook block 02: persist preflight and verify Task 2 artifacts.

This block mounts Google Drive and reads only:
  * the Block 01 JSON;
  * the exact Task 2 common-initialization file; and
  * the exact Task 2 Source-only ERM checkpoint.

It does not inspect, extract, traverse, or load PACS/Sketch data.
"""

from google.colab import drive

drive.mount("/content/drive")

import hashlib
import json
import os
import shutil
from pathlib import Path

import torch


PROTOCOL_VERSION = "task3-approved-2026-09-24-v1"

EXPECTED_RUNTIME = {
    "python": "3.13.15",
    "torch": "2.11.0+cu128",
    "torchvision": "0.26.0+cu128",
    "numpy": "2.1.3",
    "sklearn": "1.6.1",
    "cuda": "12.8",
    "cudnn": 91900,
    "device_type": "cuda",
    "gpu": "Tesla T4",
}

EXPECTED_PROTOCOL_SHA256 = (
    "e0f075e1e4f2c43c7db2423bb9b31f901d4e1157e72c2097b3e156501ce2dc74"
)
EXPECTED_INITIALIZATION_STATE_SHA256 = (
    "4d53e76c2d8f557b050a1913257c980846bebf6d5b4a28ff4d7cfa12c1d2eef3"
)
EXPECTED_ERM_CHECKPOINT_FILE_SHA256 = (
    "3d28a223e4b97b323cb3a20dcb5b7577af96631f2e6ef1f2bc99d53d85761327"
)

EXPECTED_ERM_SOURCE_VALIDATION = {
    "epoch": 4,
    "photo_accuracy": 0.9730538922155688,
    "photo_macro_f1": 0.9681339341150288,
    "art_painting_accuracy": 0.9097560975609756,
    "art_painting_macro_f1": 0.9119171075935996,
    "cartoon_accuracy": 0.9402985074626866,
    "cartoon_macro_f1": 0.947827661029101,
    "mean_source_accuracy": 0.9410361657464104,
    "mean_source_macro_f1": 0.9426262342459099,
}

LOCAL_PREFLIGHT = Path("/content/task3_runtime_preflight.json")
ATML_DRIVE_ROOT = Path("/content/drive/MyDrive/ATML-PA1")
TASK3_DRIVE_ROOT = ATML_DRIVE_ROOT / "task3_domain_generalization_20260924"
TASK3_PROVENANCE = TASK3_DRIVE_ROOT / "provenance"

TASK2_V3_ROOT = ATML_DRIVE_ROOT / "task2_corrected_normalized_v3_20260923"
COMMON_INITIALIZATION = (
    TASK2_V3_ROOT / "initialization" / "resnet18_v1_seed6304_common.pt"
)
ERM_CHECKPOINT = TASK2_V3_ROOT / "source_only" / "best.pt"


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def state_dict_sha256(state: dict[str, torch.Tensor]) -> str:
    digest = hashlib.sha256()
    for name in sorted(state):
        tensor = state[name].detach().cpu().contiguous()
        digest.update(name.encode("utf-8"))
        digest.update(str(tensor.dtype).encode("ascii"))
        digest.update(str(tuple(tensor.shape)).encode("ascii"))
        digest.update(tensor.numpy().tobytes())
    return digest.hexdigest()


def atomic_write_json(payload: dict, path: Path) -> None:
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(payload, indent=2) + "\n")
    os.replace(temporary, path)


if not LOCAL_PREFLIGHT.is_file():
    raise FileNotFoundError(
        "Block 01 JSON is missing. Rerun Block 01 in the same Colab runtime."
    )

preflight = json.loads(LOCAL_PREFLIGHT.read_text())

if preflight.get("status") != "TASK3_BLOCK_01_PASS":
    raise RuntimeError("Block 01 did not record a passing status.")
if preflight.get("protocol_version") != PROTOCOL_VERSION:
    raise RuntimeError("Block 01 protocol version differs from the approved version.")
if preflight.get("source_only_phase") is not True:
    raise RuntimeError("The source-only phase is not active.")
if preflight.get("sketch_images_accessed") != 0:
    raise RuntimeError("Block 01 reports unexpected Sketch access.")

actual_runtime = preflight.get("runtime", {})
if actual_runtime != EXPECTED_RUNTIME:
    raise RuntimeError(
        "Task 3 runtime differs from the verified Task 2 environment.\n"
        f"Expected: {json.dumps(EXPECTED_RUNTIME, sort_keys=True)}\n"
        f"Actual:   {json.dumps(actual_runtime, sort_keys=True)}"
    )

if not COMMON_INITIALIZATION.is_file():
    raise FileNotFoundError(
        "The exact Task 2 common initialization is missing:\n"
        f"{COMMON_INITIALIZATION}"
    )
if not ERM_CHECKPOINT.is_file():
    raise FileNotFoundError(
        "The exact selected Task 2 ERM checkpoint is missing:\n"
        f"{ERM_CHECKPOINT}"
    )

# Load only the two trusted, student-created Task 2 artifacts. No dataset is touched.
initialization_payload = torch.load(
    COMMON_INITIALIZATION,
    map_location="cpu",
    weights_only=False,
)

if "state_dict" not in initialization_payload:
    raise RuntimeError("Common-initialization artifact has no state_dict.")

initialization_state_sha256 = state_dict_sha256(
    initialization_payload["state_dict"]
)

if initialization_state_sha256 != EXPECTED_INITIALIZATION_STATE_SHA256:
    raise RuntimeError(
        "Common-initialization state hash mismatch.\n"
        f"Expected: {EXPECTED_INITIALIZATION_STATE_SHA256}\n"
        f"Actual:   {initialization_state_sha256}"
    )

recorded_initialization_hash = initialization_payload.get("state_dict_sha256")
if recorded_initialization_hash != EXPECTED_INITIALIZATION_STATE_SHA256:
    raise RuntimeError(
        "Common-initialization internal identity does not match the approved state."
    )

erm_checkpoint_file_sha256 = sha256_file(ERM_CHECKPOINT)
if erm_checkpoint_file_sha256 != EXPECTED_ERM_CHECKPOINT_FILE_SHA256:
    raise RuntimeError(
        "ERM checkpoint file hash mismatch.\n"
        f"Expected: {EXPECTED_ERM_CHECKPOINT_FILE_SHA256}\n"
        f"Actual:   {erm_checkpoint_file_sha256}"
    )

erm_payload = torch.load(
    ERM_CHECKPOINT,
    map_location="cpu",
    weights_only=False,
)

if erm_payload.get("target_labels_used") is not False:
    raise RuntimeError("ERM checkpoint does not certify target-label exclusion.")
if erm_payload.get("epoch") != EXPECTED_ERM_SOURCE_VALIDATION["epoch"]:
    raise RuntimeError("ERM selected epoch differs from the approved epoch.")

erm_identity = erm_payload.get("identity", {})
if erm_identity.get("protocol_sha256") != EXPECTED_PROTOCOL_SHA256:
    raise RuntimeError("ERM checkpoint uses a different source-split protocol.")
if (
    erm_identity.get("initialization_sha256")
    != EXPECTED_INITIALIZATION_STATE_SHA256
):
    raise RuntimeError("ERM checkpoint uses a different common initialization.")

erm_config = erm_identity.get("config", {})
if erm_config.get("run_id") != "source_only":
    raise RuntimeError("The supplied ERM checkpoint is not the Source-only run.")
if erm_config.get("method") != "source_only":
    raise RuntimeError("The supplied ERM checkpoint has the wrong method identity.")

actual_source_validation = {
    "epoch": erm_payload["epoch"],
    **erm_payload.get("source_validation", {}),
}

for name, expected in EXPECTED_ERM_SOURCE_VALIDATION.items():
    actual = actual_source_validation.get(name)
    if actual is None or abs(float(actual) - float(expected)) > 1e-12:
        raise RuntimeError(
            f"ERM source-validation mismatch for {name}: "
            f"expected {expected}, got {actual}"
        )

TASK3_PROVENANCE.mkdir(parents=True, exist_ok=True)

persistent_preflight = TASK3_PROVENANCE / "runtime_preflight.json"
if persistent_preflight.exists():
    if persistent_preflight.read_bytes() != LOCAL_PREFLIGHT.read_bytes():
        raise RuntimeError(
            "A different runtime_preflight.json already exists in Task 3 provenance."
        )
else:
    shutil.copy2(LOCAL_PREFLIGHT, persistent_preflight)

artifact_record = {
    "status": "TASK3_BLOCK_02_PASS",
    "protocol_version": PROTOCOL_VERSION,
    "source_only_phase": True,
    "dataset_traversed_by_this_block": False,
    "sketch_images_accessed": 0,
    "task3_drive_root": str(TASK3_DRIVE_ROOT),
    "runtime_preflight": {
        "path": str(persistent_preflight),
        "sha256": sha256_file(persistent_preflight),
    },
    "common_initialization": {
        "path": str(COMMON_INITIALIZATION),
        "state_dict_sha256": initialization_state_sha256,
        "verified": True,
    },
    "erm_checkpoint": {
        "path": str(ERM_CHECKPOINT),
        "file_sha256": erm_checkpoint_file_sha256,
        "selected_epoch": erm_payload["epoch"],
        "source_validation": erm_payload["source_validation"],
        "protocol_sha256": erm_identity["protocol_sha256"],
        "initialization_state_sha256": erm_identity["initialization_sha256"],
        "target_labels_used": erm_payload["target_labels_used"],
        "verified": True,
    },
}

artifact_record_path = TASK3_PROVENANCE / "task2_artifact_verification.json"
if artifact_record_path.exists():
    existing = json.loads(artifact_record_path.read_text())
    if existing != artifact_record:
        raise RuntimeError(
            "A different Task 2 artifact-verification record already exists."
        )
else:
    atomic_write_json(artifact_record, artifact_record_path)

print(json.dumps(artifact_record, indent=2))
print(f"\nPersisted Block 01 record: {persistent_preflight}")
print(f"Saved artifact verification: {artifact_record_path}")
print("PACS traversed: False")
print("Sketch images accessed: 0")
print("Next action: inspect this record before preparing source-only code and data.")


Mounted at /content/drive
{
  "status": "TASK3_BLOCK_02_PASS",
  "protocol_version": "task3-approved-2026-09-24-v1",
  "source_only_phase": true,
  "dataset_traversed_by_this_block": false,
  "sketch_images_accessed": 0,
  "task3_drive_root": "/content/drive/MyDrive/ATML-PA1/task3_domain_generalization_20260924",
  "runtime_preflight": {
    "path": "/content/drive/MyDrive/ATML-PA1/task3_domain_generalization_20260924/provenance/runtime_preflight.json",
    "sha256": "cee33ce799805a69eea8c484278fb3ef4d5c51f1d3bc33d6cd7d37033788c4c6"
  },
  "common_initialization": {
    "path": "/content/drive/MyDrive/ATML-PA1/task2_corrected_normalized_v3_20260923/initialization/resnet18_v1_seed6304_common.pt",
    "state_dict_sha256": "4d53e76c2d8f557b050a1913257c980846bebf6d5b4a28ff4d7cfa12c1d2eef3",
    "verified": true
  },
  "erm_checkpoint": {
    "path": "/content/drive/MyDrive/ATML-PA1/task2_corrected_normalized_v3_20260923/source_only/best.pt",
    "file_sha256": "3d28a223e4b97b323cb3a20dcb5b

In [ ]:
"""Task 3 notebook block 03: prepare a source-only PACS workspace.

Prerequisites:
  * Blocks 01 and 02 passed in this runtime.
  * Google Drive is already mounted.

This block clones the public repository only to obtain the approved Task 2 split
manifest, verifies that manifest and the PACS archive, creates a target-free protocol,
and extracts exactly the approved Photo/Art/Cartoon records. It never opens or extracts
a Sketch image member.
"""

import hashlib
import json
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path, PurePosixPath

from PIL import Image


PROTOCOL_VERSION = "task3-approved-2026-09-24-v1"
SEED = 6304
SOURCE_DOMAINS = ("photo", "art_painting", "cartoon")
FORBIDDEN_TARGET_DOMAIN = "sketch"
CLASSES = ("dog", "elephant", "giraffe", "guitar", "horse", "house", "person")
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

EXPECTED_TASK2_PROTOCOL_SHA256 = (
    "e0f075e1e4f2c43c7db2423bb9b31f901d4e1157e72c2097b3e156501ce2dc74"
)
EXPECTED_DATASET_FILE_LIST_SHA256 = (
    "559ac63b8df8e07330b97112e5ec4c414b3957585b28d21b4cfecd2181f538e0"
)
EXPECTED_PACS_ARCHIVE_SHA256 = (
    "0dc9d0176fa27c9b4504e7c2e962aebe6a79ed0c1819b84148786e590f87e102"
)
EXPECTED_ARCHIVE_MEMBERS = 10044
EXPECTED_SOURCE_COUNTS = {
    "photo": {"train": 1336, "validation": 334},
    "art_painting": {"train": 1638, "validation": 410},
    "cartoon": {"train": 1875, "validation": 469},
}
EXPECTED_SOURCE_IMAGES = sum(
    split_count
    for domain_counts in EXPECTED_SOURCE_COUNTS.values()
    for split_count in domain_counts.values()
)

REPOSITORY_URL = "https://github.com/therealshaheer11-glithc/ATML-Assignment-1.git"
CODE_ROOT = Path("/content/atml_pa1_task3_source")

ATML_DRIVE_ROOT = Path("/content/drive/MyDrive/ATML-PA1")
TASK3_DRIVE_ROOT = ATML_DRIVE_ROOT / "task3_domain_generalization_20260924"
TASK3_PROVENANCE = TASK3_DRIVE_ROOT / "provenance"
TASK3_PROTOCOL_DIR = TASK3_DRIVE_ROOT / "source_protocol"

BLOCK02_RECORD = TASK3_PROVENANCE / "task2_artifact_verification.json"
DRIVE_ARCHIVE = ATML_DRIVE_ROOT / "datasets" / "PACS_dassl.zip"

TASK2_PROTOCOL_RELATIVE = Path("shared/splits/pacs_sketch_seed6304.json")
SOURCE_PROTOCOL_PATH = TASK3_PROTOCOL_DIR / "pacs_sources_seed6304.json"
SOURCE_PREPARATION_RECORD = TASK3_PROVENANCE / "source_data_preparation.json"

SOURCE_ROOT = Path("/content/task3_pacs_sources_v1")
PARTIAL_SOURCE_ROOT = Path("/content/task3_pacs_sources_v1.partial")
LOCAL_SENTINEL = SOURCE_ROOT / "SOURCE_SNAPSHOT.json"


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def canonical_json_bytes(payload: dict) -> bytes:
    return (json.dumps(payload, indent=2, sort_keys=True) + "\n").encode("utf-8")


def atomic_write_bytes(data: bytes, path: Path) -> None:
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_bytes(data)
    os.replace(temporary, path)


def atomic_write_json(payload: dict, path: Path) -> None:
    atomic_write_bytes(canonical_json_bytes(payload), path)


def approved_relative_path(member_name: str, expected_paths: set[str]) -> str | None:
    """Map one archive member to an approved source-relative path, if applicable."""
    path = PurePosixPath(member_name)
    parts = path.parts

    if not parts or path.is_absolute() or ".." in parts:
        raise RuntimeError(f"Unsafe archive member path: {member_name}")

    lowered = tuple(part.lower() for part in parts)
    source_positions = [
        index for index, part in enumerate(lowered) if part in SOURCE_DOMAINS
    ]

    if not source_positions:
        # Metadata and non-source members are deliberately not opened.
        return None

    if len(source_positions) != 1:
        raise RuntimeError(f"Ambiguous source-domain path: {member_name}")

    source_index = source_positions[0]
    relative = PurePosixPath(*parts[source_index:]).as_posix()

    if FORBIDDEN_TARGET_DOMAIN in {part.lower() for part in parts}:
        raise RuntimeError(f"A source candidate unexpectedly contains Sketch: {member_name}")

    if relative not in expected_paths:
        return None

    return relative


def validate_task2_protocol(protocol: dict) -> set[str]:
    expected_header = {
        "dataset": "PACS",
        "seed": SEED,
        "sources": list(SOURCE_DOMAINS),
        "target": FORBIDDEN_TARGET_DOMAIN,
        "classes": list(CLASSES),
        "file_list_sha256": EXPECTED_DATASET_FILE_LIST_SHA256,
    }

    for name, expected in expected_header.items():
        if protocol.get(name) != expected:
            raise RuntimeError(
                f"Task 2 protocol field {name!r} differs: "
                f"expected {expected!r}, got {protocol.get(name)!r}"
            )

    expected_paths: set[str] = set()

    for domain in SOURCE_DOMAINS:
        for split_name in ("train", "validation"):
            rows = protocol.get("source_splits", {}).get(domain, {}).get(split_name, [])
            expected_count = EXPECTED_SOURCE_COUNTS[domain][split_name]

            if len(rows) != expected_count:
                raise RuntimeError(
                    f"Unexpected {domain}/{split_name} count: "
                    f"{len(rows)} != {expected_count}"
                )

            class_counts = [0] * len(CLASSES)

            for row in rows:
                if set(row) != {"path", "class_id"}:
                    raise RuntimeError(
                        f"Invalid source record fields in {domain}/{split_name}: {row}"
                    )

                relative = PurePosixPath(str(row["path"]))
                parts = relative.parts
                class_id = row["class_id"]

                if not parts or parts[0] != domain:
                    raise RuntimeError(f"Wrong-domain source record: {row}")
                if FORBIDDEN_TARGET_DOMAIN in {part.lower() for part in parts}:
                    raise RuntimeError(f"Sketch record found in a source split: {row}")
                if not isinstance(class_id, int) or not 0 <= class_id < len(CLASSES):
                    raise RuntimeError(f"Invalid source class ID: {row}")
                if row["path"] in expected_paths:
                    raise RuntimeError(f"Duplicate source membership: {row['path']}")

                expected_paths.add(row["path"])
                class_counts[class_id] += 1

            if any(count == 0 for count in class_counts):
                raise RuntimeError(f"A class is missing from {domain}/{split_name}")

    if len(expected_paths) != EXPECTED_SOURCE_IMAGES:
        raise RuntimeError(
            f"Expected {EXPECTED_SOURCE_IMAGES} unique source records, "
            f"found {len(expected_paths)}"
        )

    return expected_paths


def verify_local_source_snapshot(
    root: Path,
    expected_paths: set[str],
) -> tuple[str, dict[str, int]]:
    actual_paths: set[str] = set()
    counts = {domain: 0 for domain in SOURCE_DOMAINS}
    digest = hashlib.sha256()
    unreadable: list[str] = []

    for domain in SOURCE_DOMAINS:
        domain_root = root / domain
        if not domain_root.is_dir():
            raise RuntimeError(f"Missing extracted source domain: {domain_root}")

        files = sorted(
            path
            for path in domain_root.rglob("*")
            if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
        )

        for path in files:
            relative = path.relative_to(root).as_posix()
            actual_paths.add(relative)
            counts[domain] += 1
            digest.update(f"{relative}\t{path.stat().st_size}\n".encode("utf-8"))

            try:
                with Image.open(path) as image:
                    image.verify()
            except Exception:
                unreadable.append(relative)

    if unreadable:
        raise RuntimeError(f"Unreadable source images: {unreadable[:5]}")

    missing = sorted(expected_paths - actual_paths)
    unexpected = sorted(actual_paths - expected_paths)
    if missing or unexpected:
        raise RuntimeError(
            "Extracted source set differs from the approved manifest. "
            f"Missing={missing[:5]}, unexpected={unexpected[:5]}"
        )

    if len(actual_paths) != EXPECTED_SOURCE_IMAGES:
        raise RuntimeError(
            f"Unexpected extracted source count: {len(actual_paths)}"
        )

    return digest.hexdigest(), counts


if not BLOCK02_RECORD.is_file():
    raise FileNotFoundError(
        "Block 02 artifact-verification record is missing. Run Block 02 first."
    )

block02 = json.loads(BLOCK02_RECORD.read_text())
if block02.get("status") != "TASK3_BLOCK_02_PASS":
    raise RuntimeError("Block 02 did not record a passing status.")
if block02.get("source_only_phase") is not True:
    raise RuntimeError("Block 02 did not preserve the source-only phase.")
if block02.get("sketch_images_accessed") != 0:
    raise RuntimeError("Block 02 reports unexpected Sketch access.")

if not DRIVE_ARCHIVE.is_file():
    raise FileNotFoundError(f"Verified PACS archive is missing: {DRIVE_ARCHIVE}")

archive_sha256 = sha256_file(DRIVE_ARCHIVE)
if archive_sha256 != EXPECTED_PACS_ARCHIVE_SHA256:
    raise RuntimeError(
        "PACS archive hash mismatch.\n"
        f"Expected: {EXPECTED_PACS_ARCHIVE_SHA256}\n"
        f"Actual:   {archive_sha256}"
    )

if CODE_ROOT.exists():
    raise FileExistsError(
        f"Fresh clone path already exists: {CODE_ROOT}. "
        "Do not overwrite it; report this state before continuing."
    )

subprocess.run(
    ["git", "clone", "--depth", "1", REPOSITORY_URL, str(CODE_ROOT)],
    check=True,
)

repository_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=CODE_ROOT,
    text=True,
).strip()

task2_protocol_path = CODE_ROOT / TASK2_PROTOCOL_RELATIVE
if not task2_protocol_path.is_file():
    raise FileNotFoundError(
        f"Approved Task 2 split manifest is missing: {task2_protocol_path}"
    )

task2_protocol_sha256 = sha256_file(task2_protocol_path)
if task2_protocol_sha256 != EXPECTED_TASK2_PROTOCOL_SHA256:
    raise RuntimeError(
        "Task 2 split-manifest hash mismatch.\n"
        f"Expected: {EXPECTED_TASK2_PROTOCOL_SHA256}\n"
        f"Actual:   {task2_protocol_sha256}\n"
        f"Repository commit: {repository_commit}"
    )

task2_protocol = json.loads(task2_protocol_path.read_text())
expected_paths = validate_task2_protocol(task2_protocol)

source_protocol = {
    "dataset": "PACS",
    "seed": SEED,
    "sources": list(SOURCE_DOMAINS),
    "classes": list(CLASSES),
    "parent_task2_protocol_sha256": task2_protocol_sha256,
    "parent_dataset_file_list_sha256": EXPECTED_DATASET_FILE_LIST_SHA256,
    "pacs_archive_sha256": archive_sha256,
    "target_domain_embargoed": FORBIDDEN_TARGET_DOMAIN,
    "target_records_included": False,
    "source_splits": task2_protocol["source_splits"],
}

source_protocol_bytes = canonical_json_bytes(source_protocol)
source_protocol_sha256 = hashlib.sha256(source_protocol_bytes).hexdigest()

TASK3_PROTOCOL_DIR.mkdir(parents=True, exist_ok=True)
if SOURCE_PROTOCOL_PATH.exists():
    if SOURCE_PROTOCOL_PATH.read_bytes() != source_protocol_bytes:
        raise RuntimeError(
            "A different source-only protocol already exists in Task 3 Drive storage."
        )
else:
    atomic_write_bytes(source_protocol_bytes, SOURCE_PROTOCOL_PATH)

if SOURCE_ROOT.exists():
    if not LOCAL_SENTINEL.is_file():
        raise RuntimeError(
            f"Source workspace exists without a verified sentinel: {SOURCE_ROOT}"
        )

    local_record = json.loads(LOCAL_SENTINEL.read_text())
    if local_record.get("source_protocol_sha256") != source_protocol_sha256:
        raise RuntimeError("Existing source workspace uses a different protocol.")

    source_snapshot_sha256, extracted_counts = verify_local_source_snapshot(
        SOURCE_ROOT,
        expected_paths,
    )
    if source_snapshot_sha256 != local_record.get("source_snapshot_sha256"):
        raise RuntimeError("Existing source workspace snapshot hash changed.")
    reused_existing_workspace = True
else:
    if PARTIAL_SOURCE_ROOT.exists():
        raise RuntimeError(
            f"Partial source workspace already exists: {PARTIAL_SOURCE_ROOT}. "
            "Restart the runtime before retrying; do not merge partial extraction."
        )

    PARTIAL_SOURCE_ROOT.mkdir(parents=True)
    extracted_paths: set[str] = set()

    with zipfile.ZipFile(DRIVE_ARCHIVE) as archive:
        members = archive.infolist()
        if len(members) != EXPECTED_ARCHIVE_MEMBERS:
            raise RuntimeError(
                f"Unexpected PACS archive member count: {len(members)}"
            )

        for member in members:
            if member.is_dir():
                continue

            relative = approved_relative_path(member.filename, expected_paths)
            if relative is None:
                continue

            if relative in extracted_paths:
                raise RuntimeError(f"Duplicate approved archive member: {relative}")

            destination = (PARTIAL_SOURCE_ROOT / relative).resolve()
            partial_base = PARTIAL_SOURCE_ROOT.resolve()
            if partial_base not in destination.parents:
                raise RuntimeError(f"Unsafe extraction destination: {destination}")

            destination.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(member, "r") as source, destination.open("wb") as target:
                shutil.copyfileobj(source, target)

            extracted_paths.add(relative)

    missing_after_extraction = sorted(expected_paths - extracted_paths)
    if missing_after_extraction:
        raise RuntimeError(
            "Approved source members were not found in the archive: "
            f"{missing_after_extraction[:5]}"
        )
    if len(extracted_paths) != EXPECTED_SOURCE_IMAGES:
        raise RuntimeError(
            f"Extracted {len(extracted_paths)} source images; "
            f"expected {EXPECTED_SOURCE_IMAGES}"
        )

    source_snapshot_sha256, extracted_counts = verify_local_source_snapshot(
        PARTIAL_SOURCE_ROOT,
        expected_paths,
    )

    local_record = {
        "status": "VERIFIED_SOURCE_ONLY_PACS",
        "protocol_version": PROTOCOL_VERSION,
        "source_protocol_sha256": source_protocol_sha256,
        "parent_task2_protocol_sha256": task2_protocol_sha256,
        "pacs_archive_sha256": archive_sha256,
        "source_snapshot_sha256": source_snapshot_sha256,
        "source_image_count": len(extracted_paths),
        "per_domain_image_counts": extracted_counts,
        "sketch_members_opened_or_extracted": 0,
    }
    atomic_write_json(local_record, PARTIAL_SOURCE_ROOT / "SOURCE_SNAPSHOT.json")
    os.replace(PARTIAL_SOURCE_ROOT, SOURCE_ROOT)
    reused_existing_workspace = False

source_preparation_record = {
    "status": "TASK3_BLOCK_03_PASS",
    "protocol_version": PROTOCOL_VERSION,
    "source_only_phase": True,
    "repository": {
        "url": REPOSITORY_URL,
        "commit": repository_commit,
        "local_path": str(CODE_ROOT),
    },
    "task2_protocol": {
        "path": str(task2_protocol_path),
        "sha256": task2_protocol_sha256,
    },
    "source_protocol": {
        "path": str(SOURCE_PROTOCOL_PATH),
        "sha256": source_protocol_sha256,
        "target_records_included": False,
    },
    "pacs_archive": {
        "path": str(DRIVE_ARCHIVE),
        "sha256": archive_sha256,
        "member_count": EXPECTED_ARCHIVE_MEMBERS,
    },
    "source_workspace": {
        "path": str(SOURCE_ROOT),
        "source_snapshot_sha256": source_snapshot_sha256,
        "source_image_count": EXPECTED_SOURCE_IMAGES,
        "per_domain_image_counts": extracted_counts,
        "reused_existing_workspace": reused_existing_workspace,
    },
    "dataset_images_opened_for_verification": EXPECTED_SOURCE_IMAGES,
    "sketch_images_opened_or_extracted": 0,
}

if SOURCE_PREPARATION_RECORD.exists():
    existing_record = json.loads(SOURCE_PREPARATION_RECORD.read_text())
    if existing_record != source_preparation_record:
        raise RuntimeError(
            "A different source-data preparation record already exists in Drive."
        )
else:
    atomic_write_json(source_preparation_record, SOURCE_PREPARATION_RECORD)

print(json.dumps(source_preparation_record, indent=2))
print(f"\nSource-only PACS root: {SOURCE_ROOT}")
print(f"Source-only protocol: {SOURCE_PROTOCOL_PATH}")
print(f"Saved preparation record: {SOURCE_PREPARATION_RECORD}")
print(f"Approved source images verified: {EXPECTED_SOURCE_IMAGES}")
print("Sketch images opened or extracted: 0")
print("Next action: inspect this record before installing Task 3 implementation code.")



{
  "status": "TASK3_BLOCK_03_PASS",
  "protocol_version": "task3-approved-2026-09-24-v1",
  "source_only_phase": true,
  "repository": {
    "url": "https://github.com/therealshaheer11-glithc/ATML-Assignment-1.git",
    "commit": "12c9c772579d8fe8d129f6345f37043064c9c60c",
    "local_path": "/content/atml_pa1_task3_source"
  },
  "task2_protocol": {
    "path": "/content/atml_pa1_task3_source/shared/splits/pacs_sketch_seed6304.json",
    "sha256": "e0f075e1e4f2c43c7db2423bb9b31f901d4e1157e72c2097b3e156501ce2dc74"
  },
  "source_protocol": {
    "path": "/content/drive/MyDrive/ATML-PA1/task3_domain_generalization_20260924/source_protocol/pacs_sources_seed6304.json",
    "sha256": "626d8517b44ad50c0219adf49e827de6538561386791bed29a9153a589cd6abc",
    "target_records_included": false
  },
  "pacs_archive": {
    "path": "/content/drive/MyDrive/ATML-PA1/datasets/PACS_dassl.zip",
    "sha256": "0dc9d0176fa27c9b4504e7c2e962aebe6a79ed0c1819b84148786e590f87e102",
    "member_count": 10044
  

In [ ]:
# TASK 3 — BLOCK 04 BOOTSTRAP
# Pulls the committed implementation and runs the saved verification block.
# It performs no training and does not access Sketch.

from pathlib import Path
import runpy
import subprocess

CODE_ROOT = Path("/content/atml_pa1_task3_source")

if not (CODE_ROOT / ".git").is_dir():
    raise FileNotFoundError(
        f"Block 03 repository checkout is missing: {CODE_ROOT}"
    )

dirty = subprocess.check_output(
    ["git", "status", "--porcelain"],
    cwd=CODE_ROOT,
    text=True,
).strip()

if dirty:
    raise RuntimeError(
        "The Colab checkout has uncommitted changes. "
        "Do not overwrite them:\n" + dirty
    )

subprocess.run(
    ["git", "pull", "--ff-only"],
    cwd=CODE_ROOT,
    check=True,
)

block04 = (
    CODE_ROOT
    / "task3"
    / "notebook_blocks"
    / "04_install_and_verify_implementation.py"
)

if not block04.is_file():
    raise FileNotFoundError(
        "Block 04 was not found after pulling the repository. "
        "Confirm that the complete task3 directory was committed and pushed."
    )

runpy.run_path(str(block04), run_name="__main__")

test_running_statistics_freeze_but_affine_parameters_train (test_task3_core.BatchNormTests.test_running_statistics_freeze_but_affine_parameters_train) ... ok
test_every_approved_run_loads_with_the_locked_base (test_task3_core.ConfigurationTests.test_every_approved_run_loads_with_the_locked_base) ... ok
test_normalization_policies_are_not_conflated (test_task3_core.ConfigurationTests.test_normalization_policies_are_not_conflated) ... ok
test_nonpositive_median_stops_instead_of_being_clamped (test_task3_core.DAN_DGTests.test_nonpositive_median_stops_instead_of_being_clamped) ... ok
test_off_diagonal_zero_distances_are_retained (test_task3_core.DAN_DGTests.test_off_diagonal_zero_distances_are_retained) ... ok
test_pairwise_mmd_is_the_mean_of_three_locked_mmd_values (test_task3_core.DAN_DGTests.test_pairwise_mmd_is_the_mean_of_three_locked_mmd_values) ... ok
test_global_gradient_clipping_uses_l2_norm (test_task3_core.SAMTests.test_global_gradient_clipping_uses_l2_norm) ... ok
test_only_the

{
  "status": "TASK3_CODE_PREFLIGHT_PASS",
  "training_started": false,
  "source_only_phase": true,
  "sketch_images_accessed": 0,
  "runtime": {
    "python": "3.13.15",
    "torch": "2.11.0+cu128",
    "torchvision": "0.26.0+cu128",
    "numpy": "2.1.3",
    "sklearn": "1.6.1",
    "cuda": "12.8",
    "cudnn": 91900,
    "device_type": "cuda",
    "gpu": "Tesla T4"
  },
  "configs": {
    "dan_dg_0p1": {
      "protocol_version": "task3-approved-2026-09-24-v1",
      "seed": 6304,
      "weights": "ResNet18_Weights.IMAGENET1K_V1",
      "num_classes": 7,
      "feature_width": 512,
      "epochs": 30,
      "patience": 5,
      "source_batch_per_domain": 8,
      "steps_per_source_epoch": 235,
      "optimizer": "AdamW",
      "learning_rate": 0.0001,
      "weight_decay": 0.0001,
      "adam_betas": [
        0.9,
        0.999
      ],
      "adam_epsilon": 1e-08,
      "amsgrad": false,
      "adam_foreach": false,
      "adam_fused": false,
      "scheduler": null,
      "precis

{'__name__': '__main__',
 '__doc__': "Task 3 notebook block 04: install and verify the source-only implementation.\n\nPrerequisites:\n  * Blocks 01, 02, and 03 passed in this runtime.\n  * The reviewed Task 3 files have been committed to the repository's default branch.\n\nThis block fast-forwards the existing Colab checkout, runs the target-free unit tests,\nand executes the read-only code/data/artifact preflight. It does not train a model and\ndoes not traverse, extract, or open Sketch.\n",
 '__package__': '',
 '__loader__': None,
 '__spec__': None,
 '__file__': '/content/atml_pa1_task3_source/task3/notebook_blocks/04_install_and_verify_implementation.py',
 '__cached__': None,
 '__builtins__': {'__name__': 'builtins',
  '__doc__': "Built-in functions, types, exceptions, and other objects.\n\nThis module provides direct access to all 'built-in'\nidentifiers of Python; for example, builtins.len is\nthe full name for the built-in function len().\n\nThis module is not normally accessed e

In [ ]:
# TASK 3 — BLOCK 05
# Train the prescribed main DAN-DG model: lambda_DG = 1.
#
# This cell:
#   - requires the passing Block 04 records;
#   - verifies the exact repository commit and preflight identity;
#   - refuses to overwrite or silently resume an existing run;
#   - uses only Photo, Art Painting, and Cartoon;
#   - never loads Sketch.

from pathlib import Path
import hashlib
import json
import subprocess
import sys


RUN_ID = "dan_dg_1"

EXPECTED_COMMIT = "19208b4c62acb980fb3246f30e062784b90d8dfc"
EXPECTED_CODE_TREE_SHA256 = (
    "4ee16e4b2b66fa051e6571a666a935e6721681e9ac8c1325d5a494ffda528e44"
)
EXPECTED_CODE_PREFLIGHT_SHA256 = (
    "40ef37d7ae08ece5526e5588e446e5301158cc4bd444e2213cedfc0a9bf73eee"
)

CODE_ROOT = Path("/content/atml_pa1_task3_source")
SOURCE_ROOT = Path("/content/task3_pacs_sources_v1")

ATML_DRIVE_ROOT = Path("/content/drive/MyDrive/ATML-PA1")
TASK3_ROOT = (
    ATML_DRIVE_ROOT
    / "task3_domain_generalization_20260924"
)
PROVENANCE_ROOT = TASK3_ROOT / "provenance"
SOURCE_PROTOCOL = (
    TASK3_ROOT
    / "source_protocol"
    / "pacs_sources_seed6304.json"
)
TRAINING_ROOT = TASK3_ROOT / "training"
RUN_DIRECTORY = TRAINING_ROOT / RUN_ID

TASK2_ROOT = (
    ATML_DRIVE_ROOT
    / "task2_corrected_normalized_v3_20260923"
)
COMMON_INITIALIZATION = (
    TASK2_ROOT
    / "initialization"
    / "resnet18_v1_seed6304_common.pt"
)

PREREGISTRATION = (
    CODE_ROOT
    / "task3"
    / "preregistration"
    / "DAN_DG_STRENGTH_EXPECTATION.md"
)
CODE_PREFLIGHT = PROVENANCE_ROOT / "code_preflight.json"
IMPLEMENTATION_RECORD = (
    PROVENANCE_ROOT
    / "implementation_verification.json"
)


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)
    return digest.hexdigest()


# Gate 1: require the passing Block 04 record.
if not IMPLEMENTATION_RECORD.is_file():
    raise FileNotFoundError(
        f"Missing Block 04 record: {IMPLEMENTATION_RECORD}"
    )

implementation = json.loads(
    IMPLEMENTATION_RECORD.read_text()
)

required_fields = {
    "status": "TASK3_BLOCK_04_PASS",
    "protocol_version": "task3-approved-2026-09-24-v1",
    "source_only_phase": True,
    "training_started": False,
    "sketch_images_accessed": 0,
    "code_tree_sha256": EXPECTED_CODE_TREE_SHA256,
}

for field, expected in required_fields.items():
    actual = implementation.get(field)
    if actual != expected:
        raise RuntimeError(
            f"Block 04 field {field!r} differs: "
            f"{actual!r} != {expected!r}"
        )

if implementation.get("unit_tests") != {
    "count": 14,
    "passed": 14,
    "failed": 0,
}:
    raise RuntimeError(
        "Block 04 did not record all 14 tests passing"
    )


# Gate 2: verify the persistent code-preflight file.
if not CODE_PREFLIGHT.is_file():
    raise FileNotFoundError(
        f"Missing code preflight: {CODE_PREFLIGHT}"
    )

actual_preflight_sha256 = sha256_file(CODE_PREFLIGHT)

if actual_preflight_sha256 != EXPECTED_CODE_PREFLIGHT_SHA256:
    raise RuntimeError(
        "Code-preflight identity changed:\n"
        f"Expected: {EXPECTED_CODE_PREFLIGHT_SHA256}\n"
        f"Actual:   {actual_preflight_sha256}"
    )

if (
    implementation["code_preflight"]["sha256"]
    != actual_preflight_sha256
):
    raise RuntimeError(
        "Implementation record and code preflight disagree"
    )


# Gate 3: prevent code changes after Block 04.
current_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=CODE_ROOT,
    text=True,
).strip()

if current_commit != EXPECTED_COMMIT:
    raise RuntimeError(
        "Repository commit changed after Block 04:\n"
        f"Expected: {EXPECTED_COMMIT}\n"
        f"Actual:   {current_commit}"
    )

if (
    implementation["repository"]["commit"]
    != current_commit
):
    raise RuntimeError(
        "Implementation record uses a different commit"
    )


# Gate 4: never overwrite or silently resume a run.
if RUN_DIRECTORY.exists():
    if (RUN_DIRECTORY / "run.json").is_file():
        raise FileExistsError(
            "DAN-DG lambda=1 is already complete. "
            "Do not retrain it."
        )

    if (RUN_DIRECTORY / "last.pt").is_file():
        raise RuntimeError(
            "A completed-epoch checkpoint already exists. "
            "Do not overwrite or silently resume it. "
            "Send me the current state so we can use the "
            "controlled resume procedure."
        )

    raise RuntimeError(
        "An unexpected incomplete DAN-DG run directory "
        f"already exists: {RUN_DIRECTORY}"
    )


command = [
    sys.executable,
    "-m",
    "task3.train",
    "--run-id",
    RUN_ID,
    "--pacs-source-root",
    str(SOURCE_ROOT),
    "--protocol",
    str(SOURCE_PROTOCOL),
    "--initialization",
    str(COMMON_INITIALIZATION),
    "--preregistration",
    str(PREREGISTRATION),
    "--code-preflight",
    str(CODE_PREFLIGHT),
    "--output",
    str(TRAINING_ROOT),
]

print("All Block 05 pre-training gates passed.")
print("Starting DAN-DG with lambda_DG = 1.")
print("Updates per epoch: 235")
print("Maximum epochs: 30")
print("Early-stopping patience: 5")
print("Selection: mean source-validation macro-F1")
print("Sketch images accessible to training: 0")
print(
    "The first epoch record will appear after source "
    "verification and one complete training epoch.\n"
)

completed = subprocess.run(
    command,
    cwd=CODE_ROOT,
    check=True,
)

print(
    "\nDAN-DG lambda=1 training process completed "
    f"with return code {completed.returncode}."
)
print(
    "Send me the complete final output before starting "
    "another run."
)

All Block 05 pre-training gates passed.
Starting DAN-DG with lambda_DG = 1.
Updates per epoch: 235
Maximum epochs: 30
Early-stopping patience: 5
Selection: mean source-validation macro-F1
Sketch images accessible to training: 0
The first epoch record will appear after source verification and one complete training epoch.


DAN-DG lambda=1 training process completed with return code 0.
Send me the complete final output before starting another run.


In [ ]:
# TASK 3 — BLOCK 05 COMPLETION AUDIT
# Audits DAN-DG lambda=1 artifacts and writes its completion record.
# No dataset images are opened.

from pathlib import Path
import csv
import hashlib
import json
import math
import os
import subprocess

import torch


RUN_ID = "dan_dg_1"

EXPECTED_COMMIT = "19208b4c62acb980fb3246f30e062784b90d8dfc"
EXPECTED_CODE_SHA256 = (
    "4ee16e4b2b66fa051e6571a666a935e6721681e9ac8c1325d5a494ffda528e44"
)
EXPECTED_PROTOCOL_SHA256 = (
    "626d8517b44ad50c0219adf49e827de6538561386791bed29a9153a589cd6abc"
)
EXPECTED_SOURCE_SNAPSHOT_SHA256 = (
    "8ded350769ee15739f8420e755e50ff4377068a4f54ab1c0ba39d5b125e658d2"
)
EXPECTED_INITIALIZATION_SHA256 = (
    "4d53e76c2d8f557b050a1913257c980846bebf6d5b4a28ff4d7cfa12c1d2eef3"
)
EXPECTED_SHARED_MMD_SHA256 = (
    "cfe0b1d9c22d7f492ea5e8f76732fbabf21c86cb53f24759af65fda09f9bfbcc"
)
EXPECTED_CODE_PREFLIGHT_SHA256 = (
    "40ef37d7ae08ece5526e5588e446e5301158cc4bd444e2213cedfc0a9bf73eee"
)

CODE_ROOT = Path("/content/atml_pa1_task3_source")
TASK3_ROOT = Path(
    "/content/drive/MyDrive/ATML-PA1/"
    "task3_domain_generalization_20260924"
)
RUN_DIRECTORY = TASK3_ROOT / "training" / RUN_ID
PROVENANCE_ROOT = TASK3_ROOT / "provenance"

RUN_MANIFEST = RUN_DIRECTORY / "run.json"
HISTORY = RUN_DIRECTORY / "history.csv"
BEST_VALIDATION = RUN_DIRECTORY / "best_source_validation.json"
BEST_CHECKPOINT = RUN_DIRECTORY / "best.pt"
LAST_CHECKPOINT = RUN_DIRECTORY / "last.pt"
COMPLETION_RECORD = (
    PROVENANCE_ROOT
    / "dan_dg_1_training_completion.json"
)


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)
    return digest.hexdigest()


def atomic_write_json(payload, path):
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, indent=2) + "\n"
    )
    os.replace(temporary, path)


# Confirm all expected training artifacts exist.
required_files = (
    RUN_MANIFEST,
    HISTORY,
    BEST_VALIDATION,
    BEST_CHECKPOINT,
    LAST_CHECKPOINT,
)

missing = [
    str(path)
    for path in required_files
    if not path.is_file()
]

if missing:
    raise FileNotFoundError(
        f"Missing DAN-DG artifacts: {missing}"
    )


# Inspect the final run manifest.
manifest = json.loads(RUN_MANIFEST.read_text())

required_manifest_fields = {
    "status": "TASK3_RUN_COMPLETE",
    "run_id": RUN_ID,
    "steps_per_source_epoch": 235,
    "source_only_phase": True,
    "sketch_images_accessed": 0,
}

for field, expected in required_manifest_fields.items():
    actual = manifest.get(field)
    if actual != expected:
        raise RuntimeError(
            f"Manifest field {field!r} differs: "
            f"{actual!r} != {expected!r}"
        )

identity = manifest.get("identity", {})
config = identity.get("config", {})

required_config = {
    "run_id": RUN_ID,
    "method": "dan_dg",
    "mmd_lambda": 1.0,
    "sam_rho": None,
    "epochs": 30,
    "patience": 5,
    "source_batch_per_domain": 8,
    "steps_per_source_epoch": 235,
    "gradient_clipping": 20.0,
    "mmd_feature_normalization":
        "l2_per_sample_mmd_input_only",
    "adversarial_feature_normalization":
        "not_applicable_to_task3",
}

for field, expected in required_config.items():
    actual = config.get(field)
    if actual != expected:
        raise RuntimeError(
            f"Configuration field {field!r} differs: "
            f"{actual!r} != {expected!r}"
        )

required_identities = {
    "code_sha256": EXPECTED_CODE_SHA256,
    "protocol_sha256": EXPECTED_PROTOCOL_SHA256,
    "source_snapshot_sha256":
        EXPECTED_SOURCE_SNAPSHOT_SHA256,
    "initialization_sha256":
        EXPECTED_INITIALIZATION_SHA256,
    "shared_mmd_sha256":
        EXPECTED_SHARED_MMD_SHA256,
    "code_preflight_sha256":
        EXPECTED_CODE_PREFLIGHT_SHA256,
    "source_only_phase": True,
    "sketch_images_accessed": 0,
}

for field, expected in required_identities.items():
    actual = identity.get(field)
    if actual != expected:
        raise RuntimeError(
            f"Run identity {field!r} differs: "
            f"{actual!r} != {expected!r}"
        )

current_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=CODE_ROOT,
    text=True,
).strip()

if current_commit != EXPECTED_COMMIT:
    raise RuntimeError(
        "Repository commit changed during training:\n"
        f"Expected: {EXPECTED_COMMIT}\n"
        f"Actual:   {current_commit}"
    )


# Inspect every completed epoch.
with HISTORY.open(newline="") as handle:
    history = list(csv.DictReader(handle))

epochs_completed = manifest.get("epochs_completed")
best_epoch = manifest.get("best_epoch")
best_f1 = float(
    manifest.get("best_mean_source_macro_f1")
)

if len(history) != epochs_completed:
    raise RuntimeError(
        "History length differs from epochs_completed"
    )

if not 1 <= epochs_completed <= 30:
    raise RuntimeError(
        f"Invalid completed epoch count: {epochs_completed}"
    )

if [int(row["epoch"]) for row in history] != list(
    range(1, epochs_completed + 1)
):
    raise RuntimeError(
        "History epochs are missing, duplicated, or unordered"
    )

if not 1 <= best_epoch <= epochs_completed:
    raise RuntimeError(
        f"Invalid selected epoch: {best_epoch}"
    )

if not math.isfinite(best_f1):
    raise RuntimeError(
        "Best source-validation macro-F1 is non-finite"
    )

finite_columns = (
    "classification_loss",
    "mmd_loss",
    "mmd_photo__art_painting",
    "mmd_photo__cartoon",
    "mmd_art_painting__cartoon",
    "mmd_median_photo__art_painting",
    "mmd_median_photo__cartoon",
    "mmd_median_art_painting__cartoon",
    "gradient_norm",
    "gradient_norm_after_clipping",
    "gradient_clipped_fraction",
    "mean_source_accuracy",
    "mean_source_macro_f1",
    "worst_source_accuracy",
    "worst_source_macro_f1",
)

for row in history:
    epoch = int(row["epoch"])

    for column in finite_columns:
        value = float(row[column])
        if not math.isfinite(value):
            raise RuntimeError(
                f"Non-finite {column} at epoch {epoch}"
            )

    for column in (
        "mmd_median_photo__art_painting",
        "mmd_median_photo__cartoon",
        "mmd_median_art_painting__cartoon",
    ):
        if float(row[column]) <= 0:
            raise RuntimeError(
                f"Non-positive {column} at epoch {epoch}"
            )

    if float(
        row["gradient_norm_after_clipping"]
    ) > 20.001:
        raise RuntimeError(
            f"Gradient clipping limit exceeded at epoch {epoch}"
        )

    clipped_fraction = float(
        row["gradient_clipped_fraction"]
    )
    if not 0.0 <= clipped_fraction <= 1.0:
        raise RuntimeError(
            f"Invalid clipping fraction at epoch {epoch}"
        )

    for column in (
        "mean_source_accuracy",
        "mean_source_macro_f1",
        "worst_source_accuracy",
        "worst_source_macro_f1",
    ):
        if not 0.0 <= float(row[column]) <= 1.0:
            raise RuntimeError(
                f"Invalid metric {column} at epoch {epoch}"
            )


# Verify strict-improvement checkpoint selection.
history_f1 = [
    float(row["mean_source_macro_f1"])
    for row in history
]
calculated_best_f1 = max(history_f1)
calculated_best_epoch = history_f1.index(
    calculated_best_f1
) + 1

if calculated_best_epoch != best_epoch:
    raise RuntimeError(
        "Recorded best epoch violates earliest strict tie policy"
    )

if abs(calculated_best_f1 - best_f1) > 1e-12:
    raise RuntimeError(
        "Recorded best source macro-F1 differs from history"
    )

if epochs_completed < 30:
    stale_epochs = epochs_completed - best_epoch
    if stale_epochs < 5:
        raise RuntimeError(
            "Run stopped early before five stale epochs"
        )


# Verify selected-validation and checkpoint artifacts.
best_validation = json.loads(
    BEST_VALIDATION.read_text()
)

if best_validation.get("epoch") != best_epoch:
    raise RuntimeError(
        "Best-validation JSON has the wrong epoch"
    )

if abs(
    float(best_validation["mean_source_macro_f1"])
    - best_f1
) > 1e-12:
    raise RuntimeError(
        "Best-validation JSON disagrees with run.json"
    )

best_checkpoint_sha256 = sha256_file(
    BEST_CHECKPOINT
)

if (
    best_checkpoint_sha256
    != manifest.get("best_checkpoint_sha256")
):
    raise RuntimeError(
        "Selected-checkpoint hash differs from run.json"
    )

checkpoint = torch.load(
    BEST_CHECKPOINT,
    map_location="cpu",
    weights_only=False,
)

if checkpoint.get("epoch") != best_epoch:
    raise RuntimeError(
        "Selected checkpoint has the wrong epoch"
    )

if checkpoint.get("identity") != identity:
    raise RuntimeError(
        "Selected checkpoint has a different run identity"
    )

if checkpoint.get("source_only_phase") is not True:
    raise RuntimeError(
        "Selected checkpoint is not source-only"
    )

if checkpoint.get("sketch_images_accessed") != 0:
    raise RuntimeError(
        "Selected checkpoint reports Sketch access"
    )

checkpoint_f1 = float(
    checkpoint["source_validation"][
        "mean_source_macro_f1"
    ]
)

if abs(checkpoint_f1 - best_f1) > 1e-12:
    raise RuntimeError(
        "Selected checkpoint metric disagrees with manifest"
    )


# Persist the audited completion record.
completion = {
    "status": "TASK3_BLOCK_05_DAN_DG_1_PASS",
    "protocol_version":
        "task3-approved-2026-09-24-v1",
    "run_id": RUN_ID,
    "method": "dan_dg",
    "mmd_lambda": 1.0,
    "source_only_phase": True,
    "training_completed": True,
    "sketch_images_accessed": 0,
    "repository_commit": current_commit,
    "code_tree_sha256": identity["code_sha256"],
    "epochs_completed": epochs_completed,
    "best_epoch": best_epoch,
    "best_mean_source_macro_f1": best_f1,
    "best_source_validation":
        checkpoint["source_validation"],
    "best_checkpoint": {
        "path": str(BEST_CHECKPOINT),
        "sha256": best_checkpoint_sha256,
    },
    "history": {
        "path": str(HISTORY),
        "sha256": sha256_file(HISTORY),
        "rows": len(history),
    },
    "run_manifest": {
        "path": str(RUN_MANIFEST),
        "sha256": sha256_file(RUN_MANIFEST),
    },
}

if COMPLETION_RECORD.exists():
    existing = json.loads(
        COMPLETION_RECORD.read_text()
    )
    if existing != completion:
        raise RuntimeError(
            "A different DAN-DG completion record exists"
        )
else:
    atomic_write_json(
        completion,
        COMPLETION_RECORD,
    )


print("DAN-DG lambda=1 epoch summary:\n")

for row in history:
    print(
        f"epoch={int(row['epoch']):02d} "
        f"classification={float(row['classification_loss']):.6f} "
        f"mmd={float(row['mmd_loss']):.6f} "
        f"gradient={float(row['gradient_norm']):.6f} "
        f"clip_fraction={float(row['gradient_clipped_fraction']):.4f} "
        f"mean_f1={float(row['mean_source_macro_f1']):.6f} "
        f"worst_f1={float(row['worst_source_macro_f1']):.6f}"
    )

print("\nAudited completion record:\n")
print(json.dumps(completion, indent=2))
print(
    f"\nSaved at: {COMPLETION_RECORD}"
)
print("Sketch images accessed: 0")
print(
    "Do not start another run until this output is reviewed."
)

DAN-DG lambda=1 epoch summary:

epoch=01 classification=0.691308 mmd=0.368499 gradient=156.746653 clip_fraction=0.9532 mean_f1=0.737124 worst_f1=0.683601
epoch=02 classification=0.520770 mmd=0.336098 gradient=326.181449 clip_fraction=1.0000 mean_f1=0.850637 worst_f1=0.785583
epoch=03 classification=0.443482 mmd=0.322966 gradient=391.077875 clip_fraction=1.0000 mean_f1=0.869329 worst_f1=0.805197
epoch=04 classification=0.427448 mmd=0.327381 gradient=487.053995 clip_fraction=1.0000 mean_f1=0.867580 worst_f1=0.826418
epoch=05 classification=0.417255 mmd=0.323125 gradient=562.776118 clip_fraction=1.0000 mean_f1=0.806641 worst_f1=0.763007
epoch=06 classification=0.373528 mmd=0.314650 gradient=571.862531 clip_fraction=1.0000 mean_f1=0.573103 worst_f1=0.523170
epoch=07 classification=0.417296 mmd=0.323429 gradient=632.913642 clip_fraction=1.0000 mean_f1=0.708536 worst_f1=0.637735
epoch=08 classification=0.419146 mmd=0.317347 gradient=782.310502 clip_fraction=1.0000 mean_f1=0.813697 worst_f1=0

In [ ]:
# TASK 3 — DAN-DG LAMBDA=1 INSTABILITY DIAGNOSTIC
#
# Reads existing CSV histories only.
# Does not load models, datasets, or Sketch.

from pathlib import Path
import csv
import json
import math
import os


STEPS_PER_EPOCH = 235

TASK3_ROOT = Path(
    "/content/drive/MyDrive/ATML-PA1/"
    "task3_domain_generalization_20260924"
)
TASK3_HISTORY = (
    TASK3_ROOT
    / "training"
    / "dan_dg_1"
    / "history.csv"
)
DIAGNOSTIC_PATH = (
    TASK3_ROOT
    / "provenance"
    / "dan_dg_1_instability_diagnostic.json"
)

CODE_ROOT = Path("/content/atml_pa1_task3_source")
TASK2_DAN_HISTORY = (
    CODE_ROOT
    / "task2"
    / "results"
    / "training"
    / "dan_1"
    / "history.csv"
)


def read_csv(path):
    if not path.is_file():
        raise FileNotFoundError(path)

    with path.open(newline="") as handle:
        return list(csv.DictReader(handle))


def atomic_write_json(payload, path):
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, indent=2) + "\n"
    )
    os.replace(temporary, path)


task3_rows = read_csv(TASK3_HISTORY)

if len(task3_rows) != 8:
    raise RuntimeError(
        f"Expected 8 Task 3 epochs, found {len(task3_rows)}"
    )

pairs = (
    "photo__art_painting",
    "photo__cartoon",
    "art_painting__cartoon",
)

diagnostic_rows = []

print("Task 3 DAN-DG lambda=1 detailed diagnostics:\n")

for row in task3_rows:
    epoch = int(row["epoch"])
    clipped_fraction = float(
        row["gradient_clipped_fraction"]
    )
    clipped_steps = round(
        clipped_fraction * STEPS_PER_EPOCH
    )

    pair_details = {}

    for pair in pairs:
        mmd = float(row[f"mmd_{pair}"])
        median = float(row[f"mmd_median_{pair}"])
        zeros = float(
            row[f"mmd_off_diagonal_zeros_{pair}"]
        )

        if not all(
            math.isfinite(value)
            for value in (mmd, median, zeros)
        ):
            raise RuntimeError(
                f"Non-finite pair diagnostic at epoch {epoch}"
            )

        if median <= 0:
            raise RuntimeError(
                f"Non-positive bandwidth median at epoch {epoch}"
            )

        pair_details[pair] = {
            "mmd": mmd,
            "median_squared_distance": median,
            "mean_off_diagonal_zero_count": zeros,
        }

    record = {
        "epoch": epoch,
        "classification_loss": float(
            row["classification_loss"]
        ),
        "mean_pairwise_mmd": float(
            row["mmd_loss"]
        ),
        "gradient_norm_before_clipping": float(
            row["gradient_norm"]
        ),
        "gradient_norm_after_clipping": float(
            row["gradient_norm_after_clipping"]
        ),
        "clipped_fraction": clipped_fraction,
        "estimated_clipped_steps": clipped_steps,
        "mean_source_macro_f1": float(
            row["mean_source_macro_f1"]
        ),
        "worst_source_macro_f1": float(
            row["worst_source_macro_f1"]
        ),
        "pairs": pair_details,
    }

    diagnostic_rows.append(record)

    medians = " ".join(
        f"{pair}={pair_details[pair]['median_squared_distance']:.8g}"
        for pair in pairs
    )

    mmd_values = " ".join(
        f"{pair}={pair_details[pair]['mmd']:.6f}"
        for pair in pairs
    )

    zero_values = " ".join(
        f"{pair}={pair_details[pair]['mean_off_diagonal_zero_count']:.3f}"
        for pair in pairs
    )

    print(
        f"\nEpoch {epoch:02d}\n"
        f"  classification: "
        f"{record['classification_loss']:.6f}\n"
        f"  mean MMD:       "
        f"{record['mean_pairwise_mmd']:.6f}\n"
        f"  gradient:       "
        f"{record['gradient_norm_before_clipping']:.6f}"
        f" -> "
        f"{record['gradient_norm_after_clipping']:.6f}\n"
        f"  clipped steps:  "
        f"{clipped_steps}/{STEPS_PER_EPOCH}\n"
        f"  mean/worst F1:  "
        f"{record['mean_source_macro_f1']:.6f} / "
        f"{record['worst_source_macro_f1']:.6f}\n"
        f"  medians:        {medians}\n"
        f"  pair MMDs:      {mmd_values}\n"
        f"  zero counts:    {zero_values}"
    )


# Compare against Task 2 DAN lambda=1 using source-side
# training diagnostics only. No Task 2 Sketch result is read.
task2_comparison = None

if TASK2_DAN_HISTORY.is_file():
    task2_rows = read_csv(TASK2_DAN_HISTORY)

    task2_comparison = [
        {
            "epoch": int(row["epoch"]),
            "mmd_loss": float(row["mmd_loss"]),
            "median_squared_distance": float(
                row["mmd_median_squared_distance"]
            ),
            "gradient_norm_before_clipping": float(
                row["gradient_norm"]
            ),
            "gradient_norm_after_clipping": float(
                row["gradient_norm_after_clipping"]
            ),
            "clipped_fraction": float(
                row["gradient_clipped_fraction"]
            ),
            "mean_source_macro_f1": float(
                row["mean_source_macro_f1"]
            ),
        }
        for row in task2_rows
    ]

    print(
        "\n\nTask 2 DAN lambda=1 source-side reference:"
    )

    for row in task2_comparison:
        print(
            f"epoch={row['epoch']:02d} "
            f"median="
            f"{row['median_squared_distance']:.8g} "
            f"gradient="
            f"{row['gradient_norm_before_clipping']:.3f}"
            f" -> "
            f"{row['gradient_norm_after_clipping']:.3f} "
            f"clip_fraction="
            f"{row['clipped_fraction']:.4f} "
            f"mean_f1="
            f"{row['mean_source_macro_f1']:.6f}"
        )
else:
    print(
        "\nTask 2 source-side DAN history was not found; "
        "the Task 3 diagnostic remains complete."
    )


initial_medians = {
    pair: diagnostic_rows[0]["pairs"][pair][
        "median_squared_distance"
    ]
    for pair in pairs
}

final_medians = {
    pair: diagnostic_rows[-1]["pairs"][pair][
        "median_squared_distance"
    ]
    for pair in pairs
}

median_ratios = {
    pair: final_medians[pair] / initial_medians[pair]
    for pair in pairs
}

diagnostic = {
    "status":
        "TASK3_DAN_DG_1_SOURCE_ONLY_DIAGNOSTIC_COMPLETE",
    "run_id": "dan_dg_1",
    "mmd_lambda": 1.0,
    "source_only_phase": True,
    "sketch_images_accessed": 0,
    "reason": (
        "Universal or near-universal gradient clipping, "
        "rising pre-clip norms, and unstable source "
        "validation performance."
    ),
    "task3_epochs": diagnostic_rows,
    "initial_pair_medians": initial_medians,
    "final_pair_medians": final_medians,
    "final_to_initial_median_ratios": median_ratios,
    "task2_dan_1_source_only_reference":
        task2_comparison,
}

if DIAGNOSTIC_PATH.exists():
    existing = json.loads(
        DIAGNOSTIC_PATH.read_text()
    )
    if existing != diagnostic:
        raise RuntimeError(
            "A different instability diagnostic already exists"
        )
else:
    atomic_write_json(
        diagnostic,
        DIAGNOSTIC_PATH,
    )

print("\n\nMedian final/initial ratios:")
print(json.dumps(median_ratios, indent=2))

print(
    f"\nSaved diagnostic: {DIAGNOSTIC_PATH}"
)
print("Sketch images accessed: 0")
print(
    "Do not begin another run until this diagnostic "
    "has been reviewed."
)

Task 3 DAN-DG lambda=1 detailed diagnostics:


Epoch 01
  classification: 0.691308
  mean MMD:       0.368499
  gradient:       156.746653 -> 19.747558
  clipped steps:  224/235
  mean/worst F1:  0.737124 / 0.683601
  medians:        photo__art_painting=0.028412549 photo__cartoon=0.030206157 art_painting__cartoon=0.026499499
  pair MMDs:      photo__art_painting=0.365148 photo__cartoon=0.378434 art_painting__cartoon=0.361916
  zero counts:    photo__art_painting=0.000 photo__cartoon=0.000 art_painting__cartoon=0.000

Epoch 02
  classification: 0.520770
  mean MMD:       0.336098
  gradient:       326.181449 -> 20.000000
  clipped steps:  235/235
  mean/worst F1:  0.850637 / 0.785583
  medians:        photo__art_painting=0.00032886332 photo__cartoon=0.00032760673 art_painting__cartoon=0.00031700681
  pair MMDs:      photo__art_painting=0.338899 photo__cartoon=0.341642 art_painting__cartoon=0.327754
  zero counts:    photo__art_painting=0.000 photo__cartoon=0.000 art_painting__cartoon=0.

In [ ]:
"""Task 3 notebook block 06: run the approved DAN-DG lambda=0.1 study.

Paste this file's complete contents into one Colab cell. The block verifies the frozen
implementation and the completed lambda=1 evidence, persists the student's explicit
authorization, launches only lambda=0.1, audits its outputs, and then stops for review.
It never loads Sketch and never starts lambda=10 automatically.
"""

import csv
import hashlib
import json
import math
import os
import subprocess
import sys
from pathlib import Path


PROTOCOL_VERSION = "task3-approved-2026-09-24-v1"
EXPECTED_REPOSITORY_COMMIT = "19208b4c62acb980fb3246f30e062784b90d8dfc"
EXPECTED_CODE_TREE_SHA256 = (
    "4ee16e4b2b66fa051e6571a666a935e6721681e9ac8c1325d5a494ffda528e44"
)
EXPECTED_CODE_PREFLIGHT_SHA256 = (
    "40ef37d7ae08ece5526e5588e446e5301158cc4bd444e2213cedfc0a9bf73eee"
)
EXPECTED_LAMBDA_1_CHECKPOINT_SHA256 = (
    "a44bff8e519134459c2d6bf056785801f9944945d160e516b4da7f5e2752992b"
)
EXPECTED_LAMBDA_1_HISTORY_SHA256 = (
    "321e267d9b628e357811da617fc5dac07149bdcbd30db030f4c8b65c20518ed7"
)
EXPECTED_LAMBDA_1_MANIFEST_SHA256 = (
    "33eb66823509417c758095c820efebdce2b8d90b64a3eecde64e3eaecea83e2f"
)

RUN_ID = "dan_dg_0p1"
EXPECTED_LAMBDA = 0.1

CODE_ROOT = Path("/content/atml_pa1_task3_source")
SOURCE_ROOT = Path("/content/task3_pacs_sources_v1")
ATML_DRIVE_ROOT = Path("/content/drive/MyDrive/ATML-PA1")
TASK3_DRIVE_ROOT = ATML_DRIVE_ROOT / "task3_domain_generalization_20260924"
PROVENANCE_ROOT = TASK3_DRIVE_ROOT / "provenance"
SOURCE_PROTOCOL = (
    TASK3_DRIVE_ROOT / "source_protocol" / "pacs_sources_seed6304.json"
)
TRAINING_ROOT = TASK3_DRIVE_ROOT / "training"
RUN_DIRECTORY = TRAINING_ROOT / RUN_ID

TASK2_ROOT = ATML_DRIVE_ROOT / "task2_corrected_normalized_v3_20260923"
COMMON_INITIALIZATION = (
    TASK2_ROOT / "initialization" / "resnet18_v1_seed6304_common.pt"
)
PREREGISTRATION = (
    CODE_ROOT / "task3" / "preregistration" / "DAN_DG_STRENGTH_EXPECTATION.md"
)
CODE_PREFLIGHT = PROVENANCE_ROOT / "code_preflight.json"
IMPLEMENTATION_RECORD = PROVENANCE_ROOT / "implementation_verification.json"
LAMBDA_1_COMPLETION = PROVENANCE_ROOT / "dan_dg_1_training_completion.json"
LAMBDA_1_DIAGNOSTIC = PROVENANCE_ROOT / "dan_dg_1_instability_diagnostic.json"
AUTHORIZATION_RECORD = PROVENANCE_ROOT / "dan_dg_0p1_authorization.json"
COMPLETION_RECORD = PROVENANCE_ROOT / "dan_dg_0p1_training_completion.json"


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def atomic_write_json(payload: dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(payload, indent=2) + "\n")
    os.replace(temporary, path)


def load_json(path: Path, description: str) -> dict:
    if not path.is_file():
        raise FileNotFoundError(f"Missing {description}: {path}")
    payload = json.loads(path.read_text())
    if not isinstance(payload, dict):
        raise RuntimeError(f"{description} is not a JSON object: {path}")
    return payload


def require_fields(record: dict, expected: dict, description: str) -> None:
    for name, value in expected.items():
        if record.get(name) != value:
            raise RuntimeError(
                f"{description} field {name!r} differs: "
                f"{record.get(name)!r} != {value!r}"
            )


def require_recorded_artifact(entry: dict, expected_sha256: str, name: str) -> Path:
    if not isinstance(entry, dict) or not isinstance(entry.get("path"), str):
        raise RuntimeError(f"The lambda=1 completion record has no valid {name} entry")
    path = Path(entry["path"])
    if not path.is_file():
        raise FileNotFoundError(f"The recorded lambda=1 {name} is missing: {path}")
    actual = sha256_file(path)
    if entry.get("sha256") != expected_sha256 or actual != expected_sha256:
        raise RuntimeError(
            f"The lambda=1 {name} hash differs from the reviewed result"
        )
    return path


# Gate 1: exact frozen implementation and repository state.
implementation = load_json(IMPLEMENTATION_RECORD, "Block 04 implementation record")
require_fields(
    implementation,
    {
        "status": "TASK3_BLOCK_04_PASS",
        "protocol_version": PROTOCOL_VERSION,
        "source_only_phase": True,
        "training_started": False,
        "sketch_images_accessed": 0,
        "code_tree_sha256": EXPECTED_CODE_TREE_SHA256,
    },
    "Block 04 implementation record",
)
if implementation.get("unit_tests") != {"count": 14, "passed": 14, "failed": 0}:
    raise RuntimeError("Block 04 does not certify all 14 target-free tests")
if sha256_file(CODE_PREFLIGHT) != EXPECTED_CODE_PREFLIGHT_SHA256:
    raise RuntimeError("The code-preflight record changed after Block 04")
current_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=CODE_ROOT, text=True
).strip()
if current_commit != EXPECTED_REPOSITORY_COMMIT:
    raise RuntimeError(
        "The Colab repository commit changed after the approved implementation gate: "
        f"{current_commit} != {EXPECTED_REPOSITORY_COMMIT}"
    )

# Gate 2: preserve and authenticate the reviewed lambda=1 main result.
lambda_1 = load_json(LAMBDA_1_COMPLETION, "lambda=1 completion record")
require_fields(
    lambda_1,
    {
        "status": "TASK3_BLOCK_05_DAN_DG_1_PASS",
        "protocol_version": PROTOCOL_VERSION,
        "run_id": "dan_dg_1",
        "method": "dan_dg",
        "mmd_lambda": 1.0,
        "source_only_phase": True,
        "training_completed": True,
        "sketch_images_accessed": 0,
        "repository_commit": EXPECTED_REPOSITORY_COMMIT,
        "code_tree_sha256": EXPECTED_CODE_TREE_SHA256,
        "epochs_completed": 8,
        "best_epoch": 3,
        "best_mean_source_macro_f1": 0.8693286334613551,
    },
    "lambda=1 completion record",
)
require_recorded_artifact(
    lambda_1.get("best_checkpoint"),
    EXPECTED_LAMBDA_1_CHECKPOINT_SHA256,
    "selected checkpoint",
)
require_recorded_artifact(
    lambda_1.get("history"), EXPECTED_LAMBDA_1_HISTORY_SHA256, "history"
)
require_recorded_artifact(
    lambda_1.get("run_manifest"),
    EXPECTED_LAMBDA_1_MANIFEST_SHA256,
    "run manifest",
)

diagnostic = load_json(LAMBDA_1_DIAGNOSTIC, "reviewed lambda=1 diagnostic")
if diagnostic.get("source_only_phase", True) is not True:
    raise RuntimeError("The lambda=1 diagnostic does not preserve source-only status")
for key, value in diagnostic.items():
    if key.startswith("sketch_images_") and value != 0:
        raise RuntimeError(f"The lambda=1 diagnostic reports Sketch access in {key}")

# Gate 3: refuse overwrite, implicit resume, or an already completed study run.
if COMPLETION_RECORD.exists():
    raise FileExistsError(
        f"The lambda=0.1 completion record already exists: {COMPLETION_RECORD}"
    )
if RUN_DIRECTORY.exists():
    if (RUN_DIRECTORY / "run.json").is_file():
        raise FileExistsError("DAN-DG lambda=0.1 is already complete; do not rerun it")
    if (RUN_DIRECTORY / "last.pt").is_file():
        raise RuntimeError(
            "A completed-epoch lambda=0.1 checkpoint already exists. Do not silently "
            "resume it; preserve the directory and request a reviewed resume block."
        )
    raise RuntimeError(f"Unexpected lambda=0.1 run directory: {RUN_DIRECTORY}")

# Persist exactly what was approved before launching the run.
authorization = {
    "status": "TASK3_DAN_DG_0P1_AUTHORIZED",
    "protocol_version": PROTOCOL_VERSION,
    "student_approval_received": True,
    "approval_basis": "source_only_review_of_prescribed_dan_dg_lambda_1",
    "approved_next_run": RUN_ID,
    "approved_mmd_lambda": EXPECTED_LAMBDA,
    "all_other_settings_unchanged": True,
    "gradient_clipping_max_norm": 20.0,
    "lambda_1_main_result_immutable": True,
    "lambda_10_requires_lambda_0p1_completion_and_review": True,
    "modified_bandwidth_experiment_postponed_until_required_runs_complete": True,
    "source_only_phase": True,
    "sketch_images_accessed": 0,
    "repository_commit": current_commit,
    "code_tree_sha256": EXPECTED_CODE_TREE_SHA256,
    "lambda_1_completion_record": {
        "path": str(LAMBDA_1_COMPLETION),
        "sha256": sha256_file(LAMBDA_1_COMPLETION),
    },
    "lambda_1_diagnostic_record": {
        "path": str(LAMBDA_1_DIAGNOSTIC),
        "sha256": sha256_file(LAMBDA_1_DIAGNOSTIC),
    },
}
if AUTHORIZATION_RECORD.exists():
    if load_json(AUTHORIZATION_RECORD, "lambda=0.1 authorization") != authorization:
        raise RuntimeError("An existing lambda=0.1 authorization record differs")
else:
    atomic_write_json(authorization, AUTHORIZATION_RECORD)

command = [
    sys.executable,
    "-m",
    "task3.train",
    "--run-id",
    RUN_ID,
    "--pacs-source-root",
    str(SOURCE_ROOT),
    "--protocol",
    str(SOURCE_PROTOCOL),
    "--initialization",
    str(COMMON_INITIALIZATION),
    "--preregistration",
    str(PREREGISTRATION),
    "--code-preflight",
    str(CODE_PREFLIGHT),
    "--output",
    str(TRAINING_ROOT),
]

print("All Block 06 pre-training gates passed.")
print("Starting DAN-DG controlled-study run with lambda_DG = 0.1.")
print("Everything except lambda_DG is identical to the reviewed lambda=1 run.")
print("Updates per epoch: 235")
print("Maximum epochs: 30")
print("Early-stopping patience: 5")
print("Selection: unweighted mean source-validation macro-F1")
print("Gradient clipping max-norm: 20")
print("Sketch images accessible to training: 0")
print("No lambda=10 or SAM run will start from this block.\n")
subprocess.run(command, cwd=CODE_ROOT, check=True)

# Audit the completed lambda=0.1 artifacts.
run_manifest_path = RUN_DIRECTORY / "run.json"
history_path = RUN_DIRECTORY / "history.csv"
best_validation_path = RUN_DIRECTORY / "best_source_validation.json"
best_checkpoint_path = RUN_DIRECTORY / "best.pt"
run_manifest = load_json(run_manifest_path, "lambda=0.1 run manifest")
require_fields(
    run_manifest,
    {
        "status": "TASK3_RUN_COMPLETE",
        "run_id": RUN_ID,
        "source_only_phase": True,
        "sketch_images_accessed": 0,
        "steps_per_source_epoch": 235,
    },
    "lambda=0.1 run manifest",
)
identity = run_manifest.get("identity")
if not isinstance(identity, dict):
    raise RuntimeError("The lambda=0.1 run manifest has no identity")
config = identity.get("config")
if not isinstance(config, dict):
    raise RuntimeError("The lambda=0.1 run identity has no configuration")
require_fields(
    config,
    {
        "run_id": RUN_ID,
        "method": "dan_dg",
        "mmd_lambda": EXPECTED_LAMBDA,
        "gradient_clipping": 20.0,
        "source_batch_per_domain": 8,
        "steps_per_source_epoch": 235,
        "epochs": 30,
        "patience": 5,
        "selection_metric": "unweighted_mean_source_validation_macro_f1",
        "mmd_feature_normalization": "l2_per_sample_mmd_input_only",
        "mmd_bandwidth_pairs": (
            "strict_upper_triangle_keep_off_diagonal_zeros"
        ),
        "mmd_kernel": "exp(-squared_distance/(2*bandwidth))",
        "mmd_estimator": "literal_empirical_mean_embedding_v_statistic",
    },
    "lambda=0.1 configuration",
)
require_fields(
    identity,
    {
        "code_sha256": EXPECTED_CODE_TREE_SHA256,
        "code_preflight_sha256": EXPECTED_CODE_PREFLIGHT_SHA256,
        "source_only_phase": True,
        "sketch_images_accessed": 0,
    },
    "lambda=0.1 run identity",
)

epochs_completed = run_manifest.get("epochs_completed")
best_epoch = run_manifest.get("best_epoch")
best_f1 = run_manifest.get("best_mean_source_macro_f1")
if not isinstance(epochs_completed, int) or not 1 <= epochs_completed <= 30:
    raise RuntimeError("The lambda=0.1 run recorded an invalid epoch count")
if not isinstance(best_epoch, int) or not 1 <= best_epoch <= epochs_completed:
    raise RuntimeError("The lambda=0.1 run recorded an invalid best epoch")
if not isinstance(best_f1, (int, float)) or not math.isfinite(float(best_f1)):
    raise RuntimeError("The lambda=0.1 run recorded an invalid selected macro-F1")

best_validation = load_json(best_validation_path, "selected source validation")
if best_validation.get("epoch") != best_epoch:
    raise RuntimeError("Selected source-validation epoch differs from run.json")
if not math.isclose(
    float(best_validation.get("mean_source_macro_f1")),
    float(best_f1),
    rel_tol=0.0,
    abs_tol=1e-15,
):
    raise RuntimeError("Selected source macro-F1 differs from run.json")

if not best_checkpoint_path.is_file():
    raise FileNotFoundError("The selected lambda=0.1 checkpoint is missing")
best_checkpoint_sha256 = sha256_file(best_checkpoint_path)
if best_checkpoint_sha256 != run_manifest.get("best_checkpoint_sha256"):
    raise RuntimeError("The selected lambda=0.1 checkpoint hash differs from run.json")
if not history_path.is_file():
    raise FileNotFoundError("The lambda=0.1 history is missing")
with history_path.open(newline="") as handle:
    history = list(csv.DictReader(handle))
if len(history) != epochs_completed:
    raise RuntimeError("The lambda=0.1 history length differs from run.json")
if [int(row["epoch"]) for row in history] != list(range(1, epochs_completed + 1)):
    raise RuntimeError("The lambda=0.1 history epochs are not consecutive")

completion = {
    "status": "TASK3_BLOCK_06_DAN_DG_0P1_PASS",
    "protocol_version": PROTOCOL_VERSION,
    "run_id": RUN_ID,
    "method": "dan_dg",
    "mmd_lambda": EXPECTED_LAMBDA,
    "source_only_phase": True,
    "training_completed": True,
    "review_required_before_next_run": True,
    "next_run_started": False,
    "sketch_images_accessed": 0,
    "repository_commit": current_commit,
    "code_tree_sha256": identity["code_sha256"],
    "epochs_completed": epochs_completed,
    "best_epoch": best_epoch,
    "best_mean_source_macro_f1": float(best_f1),
    "best_source_validation": {
        key: value for key, value in best_validation.items() if key != "epoch"
    },
    "authorization": {
        "path": str(AUTHORIZATION_RECORD),
        "sha256": sha256_file(AUTHORIZATION_RECORD),
    },
    "best_checkpoint": {
        "path": str(best_checkpoint_path),
        "sha256": best_checkpoint_sha256,
    },
    "history": {
        "path": str(history_path),
        "sha256": sha256_file(history_path),
        "rows": len(history),
    },
    "run_manifest": {
        "path": str(run_manifest_path),
        "sha256": sha256_file(run_manifest_path),
    },
}
atomic_write_json(completion, COMPLETION_RECORD)

print("\nDAN-DG lambda=0.1 epoch summary:")
for row in history:
    print(
        f"epoch={int(row['epoch']):02d} "
        f"classification={float(row['classification_loss']):.6f} "
        f"mmd={float(row['mmd_loss']):.6f} "
        f"gradient={float(row['gradient_norm']):.6f} "
        f"clip_fraction={float(row['gradient_clipped_fraction']):.4f} "
        f"mean_f1={float(row['mean_source_macro_f1']):.6f} "
        f"worst_f1={float(row['worst_source_macro_f1']):.6f}"
    )

print("\nAudited completion record:\n")
print(json.dumps(completion, indent=2))
print(f"\nSaved at: {COMPLETION_RECORD}")
print("Sketch images accessed: 0")
print("No subsequent run was started.")
print("Send the complete final output for review before running lambda=10.")


All Block 06 pre-training gates passed.
Starting DAN-DG controlled-study run with lambda_DG = 0.1.
Everything except lambda_DG is identical to the reviewed lambda=1 run.
Updates per epoch: 235
Maximum epochs: 30
Early-stopping patience: 5
Selection: unweighted mean source-validation macro-F1
Gradient clipping max-norm: 20
Sketch images accessible to training: 0
No lambda=10 or SAM run will start from this block.


DAN-DG lambda=0.1 epoch summary:
epoch=01 classification=0.466476 mmd=0.398671 gradient=11.895829 clip_fraction=0.0511 mean_f1=0.856598 worst_f1=0.809596
epoch=02 classification=0.222693 mmd=0.355087 gradient=9.388581 clip_fraction=0.0298 mean_f1=0.913819 worst_f1=0.885467
epoch=03 classification=0.137011 mmd=0.319983 gradient=7.948457 clip_fraction=0.0298 mean_f1=0.877375 worst_f1=0.811793
epoch=04 classification=0.122875 mmd=0.327649 gradient=7.645239 clip_fraction=0.0043 mean_f1=0.937938 worst_f1=0.905946
epoch=05 classification=0.080075 mmd=0.319524 gradient=6.376280 clip

In [ ]:
"""Task 3 notebook block 07: run the approved DAN-DG lambda=10 study.

Paste this file's complete contents into one Colab cell. The block authenticates the
frozen implementation and reviewed lambda=0.1 result, records that the review condition
for lambda=10 was satisfied, launches only lambda=10, audits successful outputs, and
then stops for review. It never loads Sketch and never starts SAM automatically.
"""

import csv
import hashlib
import json
import math
import os
import subprocess
import sys
from pathlib import Path


PROTOCOL_VERSION = "task3-approved-2026-09-24-v1"
EXPECTED_REPOSITORY_COMMIT = "19208b4c62acb980fb3246f30e062784b90d8dfc"
EXPECTED_CODE_TREE_SHA256 = (
    "4ee16e4b2b66fa051e6571a666a935e6721681e9ac8c1325d5a494ffda528e44"
)
EXPECTED_CODE_PREFLIGHT_SHA256 = (
    "40ef37d7ae08ece5526e5588e446e5301158cc4bd444e2213cedfc0a9bf73eee"
)
EXPECTED_0P1_AUTHORIZATION_SHA256 = (
    "115e50f674a75660d135d9b4d9897879b15538d83ae85521f58c9206ff1e027c"
)
EXPECTED_0P1_CHECKPOINT_SHA256 = (
    "dc6036a28e3af8c281b143adb6f47b4d03676fa44c117e912ef9bcd959d8ca27"
)
EXPECTED_0P1_HISTORY_SHA256 = (
    "fc92efd81ea529a48373977f9d4973e754eb53cfca7622c8513ad5cfc64b35b8"
)
EXPECTED_0P1_MANIFEST_SHA256 = (
    "731f07bded36f6034e014eb217dca2b1b3a9c7836ed0e5bd72d0c2be3537672a"
)

RUN_ID = "dan_dg_10"
EXPECTED_LAMBDA = 10.0

CODE_ROOT = Path("/content/atml_pa1_task3_source")
SOURCE_ROOT = Path("/content/task3_pacs_sources_v1")
ATML_DRIVE_ROOT = Path("/content/drive/MyDrive/ATML-PA1")
TASK3_DRIVE_ROOT = ATML_DRIVE_ROOT / "task3_domain_generalization_20260924"
PROVENANCE_ROOT = TASK3_DRIVE_ROOT / "provenance"
SOURCE_PROTOCOL = (
    TASK3_DRIVE_ROOT / "source_protocol" / "pacs_sources_seed6304.json"
)
TRAINING_ROOT = TASK3_DRIVE_ROOT / "training"
RUN_DIRECTORY = TRAINING_ROOT / RUN_ID

TASK2_ROOT = ATML_DRIVE_ROOT / "task2_corrected_normalized_v3_20260923"
COMMON_INITIALIZATION = (
    TASK2_ROOT / "initialization" / "resnet18_v1_seed6304_common.pt"
)
PREREGISTRATION = (
    CODE_ROOT / "task3" / "preregistration" / "DAN_DG_STRENGTH_EXPECTATION.md"
)
CODE_PREFLIGHT = PROVENANCE_ROOT / "code_preflight.json"
IMPLEMENTATION_RECORD = PROVENANCE_ROOT / "implementation_verification.json"
LAMBDA_0P1_COMPLETION = PROVENANCE_ROOT / "dan_dg_0p1_training_completion.json"
AUTHORIZATION_RECORD = PROVENANCE_ROOT / "dan_dg_10_authorization.json"
COMPLETION_RECORD = PROVENANCE_ROOT / "dan_dg_10_training_completion.json"


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def atomic_write_json(payload: dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(payload, indent=2) + "\n")
    os.replace(temporary, path)


def load_json(path: Path, description: str) -> dict:
    if not path.is_file():
        raise FileNotFoundError(f"Missing {description}: {path}")
    payload = json.loads(path.read_text())
    if not isinstance(payload, dict):
        raise RuntimeError(f"{description} is not a JSON object: {path}")
    return payload


def require_fields(record: dict, expected: dict, description: str) -> None:
    for name, value in expected.items():
        if record.get(name) != value:
            raise RuntimeError(
                f"{description} field {name!r} differs: "
                f"{record.get(name)!r} != {value!r}"
            )


def require_recorded_artifact(entry: dict, expected_sha256: str, name: str) -> Path:
    if not isinstance(entry, dict) or not isinstance(entry.get("path"), str):
        raise RuntimeError(f"The lambda=0.1 record has no valid {name} entry")
    path = Path(entry["path"])
    if not path.is_file():
        raise FileNotFoundError(f"The recorded lambda=0.1 {name} is missing: {path}")
    actual = sha256_file(path)
    if entry.get("sha256") != expected_sha256 or actual != expected_sha256:
        raise RuntimeError(f"The lambda=0.1 {name} hash differs from the reviewed run")
    return path


# Gate 1: exact frozen implementation and repository state.
implementation = load_json(IMPLEMENTATION_RECORD, "Block 04 implementation record")
require_fields(
    implementation,
    {
        "status": "TASK3_BLOCK_04_PASS",
        "protocol_version": PROTOCOL_VERSION,
        "source_only_phase": True,
        "training_started": False,
        "sketch_images_accessed": 0,
        "code_tree_sha256": EXPECTED_CODE_TREE_SHA256,
    },
    "Block 04 implementation record",
)
if implementation.get("unit_tests") != {"count": 14, "passed": 14, "failed": 0}:
    raise RuntimeError("Block 04 does not certify all 14 target-free tests")
if sha256_file(CODE_PREFLIGHT) != EXPECTED_CODE_PREFLIGHT_SHA256:
    raise RuntimeError("The code-preflight record changed after Block 04")
current_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=CODE_ROOT, text=True
).strip()
if current_commit != EXPECTED_REPOSITORY_COMMIT:
    raise RuntimeError(
        "The Colab repository commit changed after the approved implementation gate: "
        f"{current_commit} != {EXPECTED_REPOSITORY_COMMIT}"
    )

# Gate 2: authenticate the completed and reviewed lambda=0.1 prerequisite.
lambda_0p1 = load_json(LAMBDA_0P1_COMPLETION, "lambda=0.1 completion record")
require_fields(
    lambda_0p1,
    {
        "status": "TASK3_BLOCK_06_DAN_DG_0P1_PASS",
        "protocol_version": PROTOCOL_VERSION,
        "run_id": "dan_dg_0p1",
        "method": "dan_dg",
        "mmd_lambda": 0.1,
        "source_only_phase": True,
        "training_completed": True,
        "review_required_before_next_run": True,
        "next_run_started": False,
        "sketch_images_accessed": 0,
        "repository_commit": EXPECTED_REPOSITORY_COMMIT,
        "code_tree_sha256": EXPECTED_CODE_TREE_SHA256,
        "epochs_completed": 10,
        "best_epoch": 5,
        "best_mean_source_macro_f1": 0.9462297763360951,
    },
    "lambda=0.1 completion record",
)
require_recorded_artifact(
    lambda_0p1.get("authorization"),
    EXPECTED_0P1_AUTHORIZATION_SHA256,
    "authorization",
)
require_recorded_artifact(
    lambda_0p1.get("best_checkpoint"),
    EXPECTED_0P1_CHECKPOINT_SHA256,
    "selected checkpoint",
)
require_recorded_artifact(
    lambda_0p1.get("history"), EXPECTED_0P1_HISTORY_SHA256, "history"
)
require_recorded_artifact(
    lambda_0p1.get("run_manifest"),
    EXPECTED_0P1_MANIFEST_SHA256,
    "run manifest",
)

# Gate 3: refuse overwrite, implicit resume, or an already completed lambda=10 run.
if COMPLETION_RECORD.exists():
    raise FileExistsError(
        f"The lambda=10 completion record already exists: {COMPLETION_RECORD}"
    )
if RUN_DIRECTORY.exists():
    if (RUN_DIRECTORY / "run.json").is_file():
        raise FileExistsError("DAN-DG lambda=10 is already complete; do not rerun it")
    if (RUN_DIRECTORY / "last.pt").is_file():
        raise RuntimeError(
            "A completed-epoch lambda=10 checkpoint already exists. Do not silently "
            "resume it; preserve the directory and request a reviewed resume block."
        )
    raise RuntimeError(f"Unexpected lambda=10 run directory: {RUN_DIRECTORY}")

# Persist the already-approved decision and the completed lambda=0.1 review gate.
authorization = {
    "status": "TASK3_DAN_DG_10_AUTHORIZED_AFTER_0P1_REVIEW",
    "protocol_version": PROTOCOL_VERSION,
    "student_approval_received": True,
    "lambda_0p1_completed_and_reviewed": True,
    "review_basis": "source_only_metrics_and_optimization_diagnostics",
    "approved_next_run": RUN_ID,
    "approved_mmd_lambda": EXPECTED_LAMBDA,
    "all_other_settings_unchanged": True,
    "gradient_clipping_max_norm": 20.0,
    "strong_alignment_instability_expected_and_preserved_if_observed": True,
    "lambda_1_main_result_immutable": True,
    "modified_bandwidth_experiment_postponed_until_required_runs_complete": True,
    "source_only_phase": True,
    "sketch_images_accessed": 0,
    "repository_commit": current_commit,
    "code_tree_sha256": EXPECTED_CODE_TREE_SHA256,
    "lambda_0p1_completion_record": {
        "path": str(LAMBDA_0P1_COMPLETION),
        "sha256": sha256_file(LAMBDA_0P1_COMPLETION),
    },
}
if AUTHORIZATION_RECORD.exists():
    if load_json(AUTHORIZATION_RECORD, "lambda=10 authorization") != authorization:
        raise RuntimeError("An existing lambda=10 authorization record differs")
else:
    atomic_write_json(authorization, AUTHORIZATION_RECORD)

command = [
    sys.executable,
    "-m",
    "task3.train",
    "--run-id",
    RUN_ID,
    "--pacs-source-root",
    str(SOURCE_ROOT),
    "--protocol",
    str(SOURCE_PROTOCOL),
    "--initialization",
    str(COMMON_INITIALIZATION),
    "--preregistration",
    str(PREREGISTRATION),
    "--code-preflight",
    str(CODE_PREFLIGHT),
    "--output",
    str(TRAINING_ROOT),
]

print("All Block 07 pre-training gates passed.")
print("The DAN-DG lambda=0.1 prerequisite completed and was reviewed.")
print("Starting DAN-DG controlled-study run with lambda_DG = 10.")
print("Everything except lambda_DG is identical to the lambda=0.1 and lambda=1 runs.")
print("Updates per epoch: 235")
print("Maximum epochs: 30")
print("Early-stopping patience: 5")
print("Selection: unweighted mean source-validation macro-F1")
print("Gradient clipping max-norm: 20")
print("Non-finite values will stop the run; finite weak results will be preserved.")
print("Sketch images accessible to training: 0")
print("No SAM run will start from this block.\n")
subprocess.run(command, cwd=CODE_ROOT, check=True)

# Audit a normally completed lambda=10 run.
run_manifest_path = RUN_DIRECTORY / "run.json"
history_path = RUN_DIRECTORY / "history.csv"
best_validation_path = RUN_DIRECTORY / "best_source_validation.json"
best_checkpoint_path = RUN_DIRECTORY / "best.pt"
run_manifest = load_json(run_manifest_path, "lambda=10 run manifest")
require_fields(
    run_manifest,
    {
        "status": "TASK3_RUN_COMPLETE",
        "run_id": RUN_ID,
        "source_only_phase": True,
        "sketch_images_accessed": 0,
        "steps_per_source_epoch": 235,
    },
    "lambda=10 run manifest",
)
identity = run_manifest.get("identity")
if not isinstance(identity, dict):
    raise RuntimeError("The lambda=10 run manifest has no identity")
config = identity.get("config")
if not isinstance(config, dict):
    raise RuntimeError("The lambda=10 run identity has no configuration")
require_fields(
    config,
    {
        "run_id": RUN_ID,
        "method": "dan_dg",
        "mmd_lambda": EXPECTED_LAMBDA,
        "gradient_clipping": 20.0,
        "source_batch_per_domain": 8,
        "steps_per_source_epoch": 235,
        "epochs": 30,
        "patience": 5,
        "selection_metric": "unweighted_mean_source_validation_macro_f1",
        "mmd_feature_normalization": "l2_per_sample_mmd_input_only",
        "mmd_bandwidth_pairs": "strict_upper_triangle_keep_off_diagonal_zeros",
        "mmd_kernel": "exp(-squared_distance/(2*bandwidth))",
        "mmd_estimator": "literal_empirical_mean_embedding_v_statistic",
    },
    "lambda=10 configuration",
)
require_fields(
    identity,
    {
        "code_sha256": EXPECTED_CODE_TREE_SHA256,
        "code_preflight_sha256": EXPECTED_CODE_PREFLIGHT_SHA256,
        "source_only_phase": True,
        "sketch_images_accessed": 0,
    },
    "lambda=10 run identity",
)

epochs_completed = run_manifest.get("epochs_completed")
best_epoch = run_manifest.get("best_epoch")
best_f1 = run_manifest.get("best_mean_source_macro_f1")
if not isinstance(epochs_completed, int) or not 1 <= epochs_completed <= 30:
    raise RuntimeError("The lambda=10 run recorded an invalid epoch count")
if not isinstance(best_epoch, int) or not 1 <= best_epoch <= epochs_completed:
    raise RuntimeError("The lambda=10 run recorded an invalid best epoch")
if not isinstance(best_f1, (int, float)) or not math.isfinite(float(best_f1)):
    raise RuntimeError("The lambda=10 run recorded an invalid selected macro-F1")

best_validation = load_json(best_validation_path, "selected source validation")
if best_validation.get("epoch") != best_epoch:
    raise RuntimeError("Selected source-validation epoch differs from run.json")
if not math.isclose(
    float(best_validation.get("mean_source_macro_f1")),
    float(best_f1),
    rel_tol=0.0,
    abs_tol=1e-15,
):
    raise RuntimeError("Selected source macro-F1 differs from run.json")

if not best_checkpoint_path.is_file():
    raise FileNotFoundError("The selected lambda=10 checkpoint is missing")
best_checkpoint_sha256 = sha256_file(best_checkpoint_path)
if best_checkpoint_sha256 != run_manifest.get("best_checkpoint_sha256"):
    raise RuntimeError("The selected lambda=10 checkpoint hash differs from run.json")
if not history_path.is_file():
    raise FileNotFoundError("The lambda=10 history is missing")
with history_path.open(newline="") as handle:
    history = list(csv.DictReader(handle))
if len(history) != epochs_completed:
    raise RuntimeError("The lambda=10 history length differs from run.json")
if [int(row["epoch"]) for row in history] != list(range(1, epochs_completed + 1)):
    raise RuntimeError("The lambda=10 history epochs are not consecutive")

completion = {
    "status": "TASK3_BLOCK_07_DAN_DG_10_PASS",
    "protocol_version": PROTOCOL_VERSION,
    "run_id": RUN_ID,
    "method": "dan_dg",
    "mmd_lambda": EXPECTED_LAMBDA,
    "source_only_phase": True,
    "training_completed": True,
    "review_required_before_next_run": True,
    "next_run_started": False,
    "sketch_images_accessed": 0,
    "repository_commit": current_commit,
    "code_tree_sha256": identity["code_sha256"],
    "epochs_completed": epochs_completed,
    "best_epoch": best_epoch,
    "best_mean_source_macro_f1": float(best_f1),
    "best_source_validation": {
        key: value for key, value in best_validation.items() if key != "epoch"
    },
    "authorization": {
        "path": str(AUTHORIZATION_RECORD),
        "sha256": sha256_file(AUTHORIZATION_RECORD),
    },
    "best_checkpoint": {
        "path": str(best_checkpoint_path),
        "sha256": best_checkpoint_sha256,
    },
    "history": {
        "path": str(history_path),
        "sha256": sha256_file(history_path),
        "rows": len(history),
    },
    "run_manifest": {
        "path": str(run_manifest_path),
        "sha256": sha256_file(run_manifest_path),
    },
}
atomic_write_json(completion, COMPLETION_RECORD)

print("\nDAN-DG lambda=10 epoch summary:")
for row in history:
    print(
        f"epoch={int(row['epoch']):02d} "
        f"classification={float(row['classification_loss']):.6f} "
        f"mmd={float(row['mmd_loss']):.6f} "
        f"gradient={float(row['gradient_norm']):.6f} "
        f"clip_fraction={float(row['gradient_clipped_fraction']):.4f} "
        f"mean_f1={float(row['mean_source_macro_f1']):.6f} "
        f"worst_f1={float(row['worst_source_macro_f1']):.6f}"
    )

print("\nAudited completion record:\n")
print(json.dumps(completion, indent=2))
print(f"\nSaved at: {COMPLETION_RECORD}")
print("Sketch images accessed: 0")
print("No subsequent run was started.")
print("Send the complete final output for review before starting SAM.")


All Block 07 pre-training gates passed.
The DAN-DG lambda=0.1 prerequisite completed and was reviewed.
Starting DAN-DG controlled-study run with lambda_DG = 10.
Everything except lambda_DG is identical to the lambda=0.1 and lambda=1 runs.
Updates per epoch: 235
Maximum epochs: 30
Early-stopping patience: 5
Selection: unweighted mean source-validation macro-F1
Gradient clipping max-norm: 20
Non-finite values will stop the run; finite weak results will be preserved.
Sketch images accessible to training: 0
No SAM run will start from this block.


DAN-DG lambda=10 epoch summary:
epoch=01 classification=2.717444 mmd=0.383395 gradient=1860.699607 clip_fraction=1.0000 mean_f1=0.036425 worst_f1=0.031238
epoch=02 classification=3.099518 mmd=0.359931 gradient=2965.230454 clip_fraction=1.0000 mean_f1=0.038202 worst_f1=0.029186
epoch=03 classification=3.726744 mmd=0.415895 gradient=3700.900336 clip_fraction=1.0000 mean_f1=0.022566 worst_f1=0.015553
epoch=04 classification=3.944545 mmd=0.379412 gra

In [ ]:
"""Task 3 notebook block 08: train the prescribed SAM main comparison.

Paste this file's complete contents into one Colab cell. It authenticates the frozen
implementation and all three completed DAN-DG conditions, launches only standard
non-adaptive SAM at rho=0.05, audits successful outputs, and stops for source-only
review. It never loads Sketch and does not begin final evaluation.
"""

import csv
import hashlib
import json
import math
import os
import subprocess
import sys
from pathlib import Path


PROTOCOL_VERSION = "task3-approved-2026-09-24-v1"
EXPECTED_REPOSITORY_COMMIT = "19208b4c62acb980fb3246f30e062784b90d8dfc"
EXPECTED_CODE_TREE_SHA256 = (
    "4ee16e4b2b66fa051e6571a666a935e6721681e9ac8c1325d5a494ffda528e44"
)
EXPECTED_CODE_PREFLIGHT_SHA256 = (
    "40ef37d7ae08ece5526e5588e446e5301158cc4bd444e2213cedfc0a9bf73eee"
)

EXPECTED_DAN_RUNS = {
    "dan_dg_0p1": {
        "record": "dan_dg_0p1_training_completion.json",
        "status": "TASK3_BLOCK_06_DAN_DG_0P1_PASS",
        "lambda": 0.1,
        "epochs": 10,
        "best_epoch": 5,
        "best_f1": 0.9462297763360951,
        "checkpoint_sha256": (
            "dc6036a28e3af8c281b143adb6f47b4d03676fa44c117e912ef9bcd959d8ca27"
        ),
        "history_sha256": (
            "fc92efd81ea529a48373977f9d4973e754eb53cfca7622c8513ad5cfc64b35b8"
        ),
        "manifest_sha256": (
            "731f07bded36f6034e014eb217dca2b1b3a9c7836ed0e5bd72d0c2be3537672a"
        ),
    },
    "dan_dg_1": {
        "record": "dan_dg_1_training_completion.json",
        "status": "TASK3_BLOCK_05_DAN_DG_1_PASS",
        "lambda": 1.0,
        "epochs": 8,
        "best_epoch": 3,
        "best_f1": 0.8693286334613551,
        "checkpoint_sha256": (
            "a44bff8e519134459c2d6bf056785801f9944945d160e516b4da7f5e2752992b"
        ),
        "history_sha256": (
            "321e267d9b628e357811da617fc5dac07149bdcbd30db030f4c8b65c20518ed7"
        ),
        "manifest_sha256": (
            "33eb66823509417c758095c820efebdce2b8d90b64a3eecde64e3eaecea83e2f"
        ),
    },
    "dan_dg_10": {
        "record": "dan_dg_10_training_completion.json",
        "status": "TASK3_BLOCK_07_DAN_DG_10_PASS",
        "lambda": 10.0,
        "epochs": 12,
        "best_epoch": 7,
        "best_f1": 0.05066996495567924,
        "checkpoint_sha256": (
            "f8cc723dc16b5e17a48f8454541beb473b70e6c38d45895f170c09f59821b2c8"
        ),
        "history_sha256": (
            "dec8773f2a0642b2aecea85619dbd8bdc19a74723c06a5ce9ca24ce9c8b4b918"
        ),
        "manifest_sha256": (
            "962626d182952f068a689bb213474074f8f07869c9a5a254cc246164d8fe4330"
        ),
    },
}

RUN_ID = "sam"
EXPECTED_RHO = 0.05

CODE_ROOT = Path("/content/atml_pa1_task3_source")
SOURCE_ROOT = Path("/content/task3_pacs_sources_v1")
ATML_DRIVE_ROOT = Path("/content/drive/MyDrive/ATML-PA1")
TASK3_DRIVE_ROOT = ATML_DRIVE_ROOT / "task3_domain_generalization_20260924"
PROVENANCE_ROOT = TASK3_DRIVE_ROOT / "provenance"
SOURCE_PROTOCOL = (
    TASK3_DRIVE_ROOT / "source_protocol" / "pacs_sources_seed6304.json"
)
TRAINING_ROOT = TASK3_DRIVE_ROOT / "training"
RUN_DIRECTORY = TRAINING_ROOT / RUN_ID

TASK2_ROOT = ATML_DRIVE_ROOT / "task2_corrected_normalized_v3_20260923"
COMMON_INITIALIZATION = (
    TASK2_ROOT / "initialization" / "resnet18_v1_seed6304_common.pt"
)
PREREGISTRATION = (
    CODE_ROOT / "task3" / "preregistration" / "DAN_DG_STRENGTH_EXPECTATION.md"
)
CODE_PREFLIGHT = PROVENANCE_ROOT / "code_preflight.json"
IMPLEMENTATION_RECORD = PROVENANCE_ROOT / "implementation_verification.json"
AUTHORIZATION_RECORD = PROVENANCE_ROOT / "sam_training_authorization.json"
COMPLETION_RECORD = PROVENANCE_ROOT / "sam_training_completion.json"


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def atomic_write_json(payload: dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(payload, indent=2) + "\n")
    os.replace(temporary, path)


def load_json(path: Path, description: str) -> dict:
    if not path.is_file():
        raise FileNotFoundError(f"Missing {description}: {path}")
    payload = json.loads(path.read_text())
    if not isinstance(payload, dict):
        raise RuntimeError(f"{description} is not a JSON object: {path}")
    return payload


def require_fields(record: dict, expected: dict, description: str) -> None:
    for name, value in expected.items():
        if record.get(name) != value:
            raise RuntimeError(
                f"{description} field {name!r} differs: "
                f"{record.get(name)!r} != {value!r}"
            )


def verify_artifact(entry: dict, expected_sha256: str, description: str) -> None:
    if not isinstance(entry, dict) or not isinstance(entry.get("path"), str):
        raise RuntimeError(f"Missing recorded {description} path")
    path = Path(entry["path"])
    if not path.is_file():
        raise FileNotFoundError(f"Recorded {description} is missing: {path}")
    actual = sha256_file(path)
    if entry.get("sha256") != expected_sha256 or actual != expected_sha256:
        raise RuntimeError(f"Recorded {description} hash differs")


# Gate 1: exact frozen implementation and repository state.
implementation = load_json(IMPLEMENTATION_RECORD, "Block 04 implementation record")
require_fields(
    implementation,
    {
        "status": "TASK3_BLOCK_04_PASS",
        "protocol_version": PROTOCOL_VERSION,
        "source_only_phase": True,
        "training_started": False,
        "sketch_images_accessed": 0,
        "code_tree_sha256": EXPECTED_CODE_TREE_SHA256,
    },
    "Block 04 implementation record",
)
if implementation.get("unit_tests") != {"count": 14, "passed": 14, "failed": 0}:
    raise RuntimeError("Block 04 does not certify all 14 target-free tests")
if sha256_file(CODE_PREFLIGHT) != EXPECTED_CODE_PREFLIGHT_SHA256:
    raise RuntimeError("The code-preflight record changed after Block 04")
current_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=CODE_ROOT, text=True
).strip()
if current_commit != EXPECTED_REPOSITORY_COMMIT:
    raise RuntimeError(
        "The Colab repository commit changed after the approved implementation gate: "
        f"{current_commit} != {EXPECTED_REPOSITORY_COMMIT}"
    )

# Gate 2: authenticate all completed DAN-DG training conditions.
dan_record_identities = {}
for run_id, expected in EXPECTED_DAN_RUNS.items():
    record_path = PROVENANCE_ROOT / expected["record"]
    record = load_json(record_path, f"{run_id} completion record")
    require_fields(
        record,
        {
            "status": expected["status"],
            "protocol_version": PROTOCOL_VERSION,
            "run_id": run_id,
            "method": "dan_dg",
            "mmd_lambda": expected["lambda"],
            "source_only_phase": True,
            "training_completed": True,
            "sketch_images_accessed": 0,
            "repository_commit": EXPECTED_REPOSITORY_COMMIT,
            "code_tree_sha256": EXPECTED_CODE_TREE_SHA256,
            "epochs_completed": expected["epochs"],
            "best_epoch": expected["best_epoch"],
            "best_mean_source_macro_f1": expected["best_f1"],
        },
        f"{run_id} completion record",
    )
    verify_artifact(
        record.get("best_checkpoint"),
        expected["checkpoint_sha256"],
        f"{run_id} selected checkpoint",
    )
    verify_artifact(
        record.get("history"), expected["history_sha256"], f"{run_id} history"
    )
    verify_artifact(
        record.get("run_manifest"),
        expected["manifest_sha256"],
        f"{run_id} run manifest",
    )
    dan_record_identities[run_id] = {
        "path": str(record_path),
        "sha256": sha256_file(record_path),
    }

# Gate 3: refuse overwrite, implicit resume, or an already completed SAM run.
if COMPLETION_RECORD.exists():
    raise FileExistsError(f"The SAM completion record already exists: {COMPLETION_RECORD}")
if RUN_DIRECTORY.exists():
    if (RUN_DIRECTORY / "run.json").is_file():
        raise FileExistsError("The SAM main run is already complete; do not rerun it")
    if (RUN_DIRECTORY / "last.pt").is_file():
        raise RuntimeError(
            "A completed-epoch SAM checkpoint already exists. Do not silently resume "
            "it; preserve the directory and request a reviewed resume block."
        )
    raise RuntimeError(f"Unexpected SAM run directory: {RUN_DIRECTORY}")

# Persist the approved fixed SAM design before launching it.
authorization = {
    "status": "TASK3_SAM_MAIN_AUTHORIZED",
    "protocol_version": PROTOCOL_VERSION,
    "student_approval_received": True,
    "dan_dg_strength_study_completed_and_reviewed": True,
    "approved_next_run": RUN_ID,
    "method": "standard_non_adaptive_sam",
    "sam_rho": EXPECTED_RHO,
    "objective": "source_erm_classification_only",
    "first_pass_gradient_clipping": False,
    "second_pass_update_gradient_clipping_max_norm": 20.0,
    "batchnorm_running_statistics_frozen_during_both_passes": True,
    "all_shared_settings_unchanged": True,
    "source_only_phase": True,
    "sketch_images_accessed": 0,
    "repository_commit": current_commit,
    "code_tree_sha256": EXPECTED_CODE_TREE_SHA256,
    "completed_dan_dg_records": dan_record_identities,
}
if AUTHORIZATION_RECORD.exists():
    if load_json(AUTHORIZATION_RECORD, "SAM authorization") != authorization:
        raise RuntimeError("An existing SAM authorization record differs")
else:
    atomic_write_json(authorization, AUTHORIZATION_RECORD)

command = [
    sys.executable,
    "-m",
    "task3.train",
    "--run-id",
    RUN_ID,
    "--pacs-source-root",
    str(SOURCE_ROOT),
    "--protocol",
    str(SOURCE_PROTOCOL),
    "--initialization",
    str(COMMON_INITIALIZATION),
    "--preregistration",
    str(PREREGISTRATION),
    "--code-preflight",
    str(CODE_PREFLIGHT),
    "--output",
    str(TRAINING_ROOT),
]

print("All Block 08 pre-training gates passed.")
print("All three DAN-DG conditions are complete and authenticated.")
print("Starting prescribed standard non-adaptive SAM main run with rho = 0.05.")
print("Objective: source ERM classification loss only")
print("Updates per epoch: 235, with two forward/backward passes per update")
print("Maximum epochs: 30")
print("Early-stopping patience: 5")
print("Selection: unweighted mean source-validation macro-F1")
print("First-pass gradient clipping: disabled")
print("Second-pass update-gradient clipping max-norm: 20")
print("BatchNorm running statistics frozen during both passes: yes")
print("Sketch images accessible to training: 0")
print("No final evaluation will start from this block.\n")
subprocess.run(command, cwd=CODE_ROOT, check=True)

# Audit a normally completed SAM run.
run_manifest_path = RUN_DIRECTORY / "run.json"
history_path = RUN_DIRECTORY / "history.csv"
best_validation_path = RUN_DIRECTORY / "best_source_validation.json"
best_checkpoint_path = RUN_DIRECTORY / "best.pt"
run_manifest = load_json(run_manifest_path, "SAM run manifest")
require_fields(
    run_manifest,
    {
        "status": "TASK3_RUN_COMPLETE",
        "run_id": RUN_ID,
        "source_only_phase": True,
        "sketch_images_accessed": 0,
        "steps_per_source_epoch": 235,
    },
    "SAM run manifest",
)
identity = run_manifest.get("identity")
if not isinstance(identity, dict):
    raise RuntimeError("The SAM run manifest has no identity")
config = identity.get("config")
if not isinstance(config, dict):
    raise RuntimeError("The SAM run identity has no configuration")
require_fields(
    config,
    {
        "run_id": RUN_ID,
        "method": "sam",
        "mmd_lambda": None,
        "sam_rho": EXPECTED_RHO,
        "sam_variant": "standard_non_adaptive",
        "sam_first_pass_clipping": False,
        "sam_second_pass_clipping": True,
        "gradient_clipping": 20.0,
        "source_batch_per_domain": 8,
        "steps_per_source_epoch": 235,
        "epochs": 30,
        "patience": 5,
        "selection_metric": "unweighted_mean_source_validation_macro_f1",
        "batchnorm_running_statistics": "frozen_at_imagenet_values",
        "batchnorm_affine_parameters": "trainable",
    },
    "SAM configuration",
)
require_fields(
    identity,
    {
        "code_sha256": EXPECTED_CODE_TREE_SHA256,
        "code_preflight_sha256": EXPECTED_CODE_PREFLIGHT_SHA256,
        "source_only_phase": True,
        "sketch_images_accessed": 0,
    },
    "SAM run identity",
)

epochs_completed = run_manifest.get("epochs_completed")
best_epoch = run_manifest.get("best_epoch")
best_f1 = run_manifest.get("best_mean_source_macro_f1")
if not isinstance(epochs_completed, int) or not 1 <= epochs_completed <= 30:
    raise RuntimeError("The SAM run recorded an invalid epoch count")
if not isinstance(best_epoch, int) or not 1 <= best_epoch <= epochs_completed:
    raise RuntimeError("The SAM run recorded an invalid best epoch")
if not isinstance(best_f1, (int, float)) or not math.isfinite(float(best_f1)):
    raise RuntimeError("The SAM run recorded an invalid selected macro-F1")

best_validation = load_json(best_validation_path, "selected SAM source validation")
if best_validation.get("epoch") != best_epoch:
    raise RuntimeError("Selected SAM source-validation epoch differs from run.json")
if not math.isclose(
    float(best_validation.get("mean_source_macro_f1")),
    float(best_f1),
    rel_tol=0.0,
    abs_tol=1e-15,
):
    raise RuntimeError("Selected SAM source macro-F1 differs from run.json")

if not best_checkpoint_path.is_file():
    raise FileNotFoundError("The selected SAM checkpoint is missing")
best_checkpoint_sha256 = sha256_file(best_checkpoint_path)
if best_checkpoint_sha256 != run_manifest.get("best_checkpoint_sha256"):
    raise RuntimeError("The selected SAM checkpoint hash differs from run.json")
if not history_path.is_file():
    raise FileNotFoundError("The SAM history is missing")
with history_path.open(newline="") as handle:
    history = list(csv.DictReader(handle))
if len(history) != epochs_completed:
    raise RuntimeError("The SAM history length differs from run.json")
if [int(row["epoch"]) for row in history] != list(range(1, epochs_completed + 1)):
    raise RuntimeError("The SAM history epochs are not consecutive")

completion = {
    "status": "TASK3_BLOCK_08_SAM_PASS",
    "protocol_version": PROTOCOL_VERSION,
    "run_id": RUN_ID,
    "method": "sam",
    "sam_variant": "standard_non_adaptive",
    "sam_rho": EXPECTED_RHO,
    "source_only_phase": True,
    "training_completed": True,
    "review_required_before_diagnostics": True,
    "final_evaluation_started": False,
    "sketch_images_accessed": 0,
    "repository_commit": current_commit,
    "code_tree_sha256": identity["code_sha256"],
    "epochs_completed": epochs_completed,
    "best_epoch": best_epoch,
    "best_mean_source_macro_f1": float(best_f1),
    "best_source_validation": {
        key: value for key, value in best_validation.items() if key != "epoch"
    },
    "authorization": {
        "path": str(AUTHORIZATION_RECORD),
        "sha256": sha256_file(AUTHORIZATION_RECORD),
    },
    "best_checkpoint": {
        "path": str(best_checkpoint_path),
        "sha256": best_checkpoint_sha256,
    },
    "history": {
        "path": str(history_path),
        "sha256": sha256_file(history_path),
        "rows": len(history),
    },
    "run_manifest": {
        "path": str(run_manifest_path),
        "sha256": sha256_file(run_manifest_path),
    },
}
atomic_write_json(completion, COMPLETION_RECORD)

print("\nSAM epoch summary:")
for row in history:
    print(
        f"epoch={int(row['epoch']):02d} "
        f"base_classification={float(row['classification_loss']):.6f} "
        f"perturbed_classification="
        f"{float(row['sam_perturbed_classification_loss']):.6f} "
        f"first_gradient={float(row['sam_first_gradient_norm']):.6f} "
        f"perturbation={float(row['sam_perturbation_norm']):.6f} "
        f"update_gradient={float(row['gradient_norm']):.6f} "
        f"clip_fraction={float(row['gradient_clipped_fraction']):.4f} "
        f"mean_f1={float(row['mean_source_macro_f1']):.6f} "
        f"worst_f1={float(row['worst_source_macro_f1']):.6f}"
    )

print("\nAudited completion record:\n")
print(json.dumps(completion, indent=2))
print(f"\nSaved at: {COMPLETION_RECORD}")
print("Sketch images accessed: 0")
print("Final evaluation started: False")
print("Send the complete final output for source-only review before diagnostics.")


All Block 08 pre-training gates passed.
All three DAN-DG conditions are complete and authenticated.
Starting prescribed standard non-adaptive SAM main run with rho = 0.05.
Objective: source ERM classification loss only
Updates per epoch: 235, with two forward/backward passes per update
Maximum epochs: 30
Early-stopping patience: 5
Selection: unweighted mean source-validation macro-F1
First-pass gradient clipping: disabled
Second-pass update-gradient clipping max-norm: 20
BatchNorm running statistics frozen during both passes: yes
Sketch images accessible to training: 0
No final evaluation will start from this block.


SAM epoch summary:
epoch=01 base_classification=0.753476 perturbed_classification=1.063220 first_gradient=5.444643 perturbation=0.050000 update_gradient=7.728617 clip_fraction=0.0170 mean_f1=0.889132 worst_f1=0.845549
epoch=02 base_classification=0.257415 perturbed_classification=0.512306 first_gradient=4.248339 perturbation=0.050000 update_gradient=6.526016 clip_fraction

In [ ]:
# TASK 3 — BLOCK 09 BOOTSTRAP
# Pulls the committed diagnostic implementation and runs source-only diagnostics.
# Does not access Sketch, create the experiment lock, or start final evaluation.

from pathlib import Path
import runpy
import subprocess

repo_root = Path("/content/atml_pa1_task3_source")

if not (repo_root / ".git").is_dir():
    raise FileNotFoundError(f"Repository checkout not found: {repo_root}")

subprocess.run(
    ["git", "pull", "--ff-only"],
    cwd=repo_root,
    check=True,
)

block09 = (
    repo_root
    / "task3"
    / "notebook_blocks"
    / "09_install_and_run_source_diagnostics.py"
)

if not block09.is_file():
    raise FileNotFoundError(f"Block 09 not found after pull: {block09}")

print(
    "Repository HEAD:",
    subprocess.check_output(
        ["git", "rev-parse", "HEAD"],
        cwd=repo_root,
        text=True,
    ).strip(),
)
print("Running:", block09)

runpy.run_path(str(block09), run_name="__main__")

Repository HEAD: 10e3c1e80a267ffe77456814071fa7895ac2743d
Running: /content/atml_pa1_task3_source/task3/notebook_blocks/09_install_and_run_source_diagnostics.py
All Block 09 pre-diagnostic gates passed.
Training commit preserved: 19208b4c62acb980fb3246f30e062784b90d8dfc
Diagnostic commit: 10e3c1e80a267ffe77456814071fa7895ac2743d
Target-free tests passed: 18/18
Probe: 334 validation examples per source, 1,002 total
Probe features: raw unnormalized 512-D pre-classifier features
Probe split: shared domain-stratified 70/30 split, seed 6304
Sharpness batch: 32 validation examples per source, 96 total
Sharpness radius: 0.05
Models: ERM, prescribed DAN-DG lambda=1, SAM
Sketch images accessible to diagnostics: 0


Audited source-only diagnostic summary:

{
  "status": "TASK3_BLOCK_09_SOURCE_DIAGNOSTICS_PASS",
  "protocol_version": "task3-approved-2026-09-24-v1",
  "source_only_phase": true,
  "diagnostics_completed": true,
  "experiment_lock_created": false,
  "final_evaluation_started": false

{'__name__': '__main__',
 '__doc__': 'Task 3 notebook block 09: install and run locked source-only diagnostics.\n\nRun only after the listed post-training files have been committed through GitHub web.\nThis block fast-forwards the clean Colab checkout, verifies the exact allowed diff and\ndiagnostic code tree, runs all 18 target-free tests, authenticates the completed SAM\nrun, executes the source-domain probe and common sharpness proxy, and persists an audit.\nIt never loads Sketch and does not create the final experiment lock.\n',
 '__package__': '',
 '__loader__': None,
 '__spec__': None,
 '__file__': '/content/atml_pa1_task3_source/task3/notebook_blocks/09_install_and_run_source_diagnostics.py',
 '__cached__': None,
 '__builtins__': {'__name__': 'builtins',
  '__doc__': "Built-in functions, types, exceptions, and other objects.\n\nThis module provides direct access to all 'built-in'\nidentifiers of Python; for example, builtins.len is\nthe full name for the built-in function len().

In [ ]:
from pathlib import Path
import runpy
import subprocess

CODE_ROOT = Path("/content/atml_pa1_task3_source")

if not (CODE_ROOT / ".git").is_dir():
    raise FileNotFoundError(f"Repository checkout not found: {CODE_ROOT}")

dirty = subprocess.check_output(
    ["git", "status", "--porcelain"],
    cwd=CODE_ROOT,
    text=True,
).strip()

if dirty:
    raise RuntimeError(
        "The Colab repository has uncommitted changes:\n" + dirty
    )

subprocess.run(
    ["git", "pull", "--ff-only"],
    cwd=CODE_ROOT,
    check=True,
)

head = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=CODE_ROOT,
    text=True,
).strip()

block10 = (
    CODE_ROOT
    / "task3"
    / "notebook_blocks"
    / "10_prepare_bandwidth_floor_research_variant.py"
)

if not block10.is_file():
    raise FileNotFoundError(block10)

print("Repository HEAD:", head)
print("Running:", block10)

runpy.run_path(str(block10), run_name="__main__")

FileNotFoundError: Repository checkout not found: /content/atml_pa1_task3_source

In [ ]:
from google.colab import drive

import hashlib
import json
import os
import runpy
import shutil
import subprocess
import sys
import zipfile

from pathlib import Path, PurePosixPath


drive.mount("/content/drive")

REPOSITORY_URL = (
    "https://github.com/therealshaheer11-glithc/"
    "ATML-Assignment-1.git"
)

CODE_ROOT = Path("/content/atml_pa1_task3_source")
SOURCE_ROOT = Path("/content/task3_pacs_sources_v1")
PARTIAL_ROOT = Path("/content/task3_pacs_sources_v1.partial")

ATML_ROOT = Path("/content/drive/MyDrive/ATML-PA1")
TASK3_ROOT = ATML_ROOT / "task3_domain_generalization_20260924"

SOURCE_PROTOCOL = (
    TASK3_ROOT
    / "source_protocol"
    / "pacs_sources_seed6304.json"
)

PACS_ARCHIVE = (
    ATML_ROOT
    / "datasets"
    / "PACS_dassl.zip"
)

EXPECTED_PROTOCOL_SHA256 = (
    "626d8517b44ad50c0219adf49e827de"
    "6538561386791bed29a9153a589cd6abc"
)

EXPECTED_ARCHIVE_SHA256 = (
    "0dc9d0176fa27c9b4504e7c2e962ae"
    "be6a79ed0c1819b84148786e590f87e102"
)

SOURCE_DOMAINS = (
    "photo",
    "art_painting",
    "cartoon",
)


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)
    return digest.hexdigest()


# Restore the current GitHub repository.
if CODE_ROOT.exists():
    if not (CODE_ROOT / ".git").is_dir():
        raise RuntimeError(
            f"Unexpected non-repository path exists: {CODE_ROOT}"
        )
else:
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            REPOSITORY_URL,
            str(CODE_ROOT),
        ],
        check=True,
    )

subprocess.run(
    ["git", "pull", "--ff-only"],
    cwd=CODE_ROOT,
    check=True,
)

head = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=CODE_ROOT,
    text=True,
).strip()

print("Repository restored at:", head)


# Verify persistent source inputs.
if not SOURCE_PROTOCOL.is_file():
    raise FileNotFoundError(SOURCE_PROTOCOL)

if sha256_file(SOURCE_PROTOCOL) != EXPECTED_PROTOCOL_SHA256:
    raise RuntimeError("Source-only protocol hash differs")

if not PACS_ARCHIVE.is_file():
    raise FileNotFoundError(PACS_ARCHIVE)

if sha256_file(PACS_ARCHIVE) != EXPECTED_ARCHIVE_SHA256:
    raise RuntimeError("PACS archive hash differs")

protocol = json.loads(SOURCE_PROTOCOL.read_text())

expected_paths = {
    record["path"]
    for domain in SOURCE_DOMAINS
    for split in ("train", "validation")
    for record in protocol["source_splits"][domain][split]
}

if len(expected_paths) != 6062:
    raise RuntimeError(
        f"Expected 6062 source records, found {len(expected_paths)}"
    )


# Reconstruct only Photo, Art Painting and Cartoon when needed.
if not SOURCE_ROOT.exists():
    if PARTIAL_ROOT.exists():
        raise RuntimeError(
            f"Partial extraction already exists: {PARTIAL_ROOT}"
        )

    PARTIAL_ROOT.mkdir(parents=True)
    extracted = set()

    with zipfile.ZipFile(PACS_ARCHIVE) as archive:
        for member in archive.infolist():
            if member.is_dir():
                continue

            path = PurePosixPath(member.filename)

            if path.is_absolute() or ".." in path.parts:
                raise RuntimeError(
                    f"Unsafe archive member: {member.filename}"
                )

            lowered = tuple(
                component.lower()
                for component in path.parts
            )

            positions = [
                index
                for index, component in enumerate(lowered)
                if component in SOURCE_DOMAINS
            ]

            if not positions:
                # This includes Sketch and metadata.
                # Those members are never opened.
                continue

            if len(positions) != 1:
                raise RuntimeError(
                    f"Ambiguous source member: {member.filename}"
                )

            if "sketch" in lowered:
                raise RuntimeError(
                    f"Source candidate contains Sketch: {member.filename}"
                )

            relative = PurePosixPath(
                *path.parts[positions[0]:]
            ).as_posix()

            if relative not in expected_paths:
                continue

            if relative in extracted:
                raise RuntimeError(
                    f"Duplicate source member: {relative}"
                )

            destination = (
                PARTIAL_ROOT / relative
            ).resolve()

            if PARTIAL_ROOT.resolve() not in destination.parents:
                raise RuntimeError(
                    f"Unsafe destination: {destination}"
                )

            destination.parent.mkdir(
                parents=True,
                exist_ok=True,
            )

            with archive.open(member) as source:
                with destination.open("wb") as target:
                    shutil.copyfileobj(source, target)

            extracted.add(relative)

    if extracted != expected_paths:
        raise RuntimeError(
            "Extracted source membership differs from protocol"
        )

    os.replace(PARTIAL_ROOT, SOURCE_ROOT)

print("Source-only workspace restored:", SOURCE_ROOT)


# Perform the repository's full source-only verification.
sys.path.insert(0, str(CODE_ROOT))

from task3.data import (  # noqa: E402
    load_source_protocol,
    verify_source_snapshot,
)

verified_protocol = load_source_protocol(
    SOURCE_PROTOCOL
)

source_verification = verify_source_snapshot(
    SOURCE_ROOT,
    verified_protocol,
)

print(
    "Verified source images:",
    source_verification["source_image_count"],
)
print(
    "Sketch images accessed:",
    source_verification["sketch_images_accessed"],
)


# Run only Block 10: tests and floor calibration, no training.
block10 = (
    CODE_ROOT
    / "task3"
    / "notebook_blocks"
    / "10_prepare_bandwidth_floor_research_variant.py"
)

if not block10.is_file():
    raise FileNotFoundError(block10)

print("Running:", block10)

runpy.run_path(
    str(block10),
    run_name="__main__",
)

Mounted at /content/drive
Repository restored at: 17b6b44dde5309dfa4249be91a35f58e902908b6
Source-only workspace restored: /content/task3_pacs_sources_v1
Verified source images: 6062
Sketch images accessed: 0


FileNotFoundError: /content/atml_pa1_task3_source/task3/notebook_blocks/10_prepare_bandwidth_floor_research_variant.py

In [ ]:
from pathlib import Path
import subprocess

repo = Path("/content/atml_pa1_task3_source")

print("Current commit:")
print(subprocess.check_output(
    ["git", "-C", str(repo), "rev-parse", "HEAD"],
    text=True,
).strip())

print("\nNotebook blocks currently committed:")
blocks = repo / "task3" / "notebook_blocks"
for path in sorted(blocks.glob("*")):
    print(path.name)

print("\nPossible misplaced Block 10 files:")
matches = list(repo.rglob("*10*bandwidth*floor*"))
if matches:
    for path in matches:
        print(path.relative_to(repo))
else:
    print("NONE")

Current commit:
17b6b44dde5309dfa4249be91a35f58e902908b6

Notebook blocks currently committed:
01_environment_preflight.py
02_mount_and_verify_task2_artifacts.py
03_prepare_source_only_data.py
04_install_and_verify_implementation.py
05a_train_dan_dg_1.py
05b_audit_dan_dg_1.py
05c_diagnose_dan_dg_1_instability.py
06_train_dan_dg_0p1.py
07_train_dan_dg_10.py
08_train_sam.py
09_install_and_run_source_diagnostics.py

Possible misplaced Block 10 files:
task3 /notebook_blocks/10_prepare_bandwidth_floor_research_variant.py


In [ ]:
from pathlib import Path

repo = Path("/content/atml_pa1_task3_source")
wrong_root = repo / "task3 "

print("Top-level names:")
for path in sorted(repo.iterdir()):
    print(repr(path.name))

print("\nFiles inside the mistaken 'task3 ' folder:")
if wrong_root.is_dir():
    for path in sorted(wrong_root.rglob("*")):
        if path.is_file():
            print(path.relative_to(wrong_root))
else:
    print("Mistaken folder not found.")

Top-level names:
'.git'
'.gitignore'
'README.md'
'colab-environment.json'
'requirements-colab.txt'
'shared'
'task1'
'task2'
'task3'
'task3 '
'tests'
'tools'

Files inside the mistaken 'task3 ' folder:
configs/dan_dg_floor_0p1.json
configs/dan_dg_floor_1.json
configs/dan_dg_floor_10.json
docs/DAN_DG_BANDWIDTH_FLOOR_RESEARCH_VARIANT.md
notebook_blocks/10_prepare_bandwidth_floor_research_variant.py
notebook_blocks/11_train_dan_dg_floor_1.py
notebook_blocks/12_train_dan_dg_floor_0p1.py
notebook_blocks/13_train_dan_dg_floor_10.py
preregistration/DAN_DG_BANDWIDTH_FLOOR_STUDY.md
provenance/RUN_LOG.md
research_variants/__init__.py
research_variants/bandwidth_floor.py
research_variants/config.py
research_variants/notebook_runner.py
tests/test_task3_bandwidth_floor.py


In [ ]:
from pathlib import Path
import runpy
import subprocess

REPO = Path("/content/atml_pa1_task3_source")
DIAGNOSTIC_COMMIT = "10e3c1e80a267ffe77456814071fa7895ac2743d"

# Obtain the path-correction commit.
subprocess.run(
    ["git", "-C", str(REPO), "pull", "--ff-only"],
    check=True,
)

EXPECTED_FILES = {
    "task3/README.md",
    "task3/calibrate_bandwidth_floor.py",
    "task3/train_bandwidth_floor.py",
    "task3/configs/dan_dg_floor_0p1.json",
    "task3/configs/dan_dg_floor_1.json",
    "task3/configs/dan_dg_floor_10.json",
    "task3/docs/DAN_DG_BANDWIDTH_FLOOR_RESEARCH_VARIANT.md",
    "task3/notebook_blocks/10_prepare_bandwidth_floor_research_variant.py",
    "task3/notebook_blocks/11_train_dan_dg_floor_1.py",
    "task3/notebook_blocks/12_train_dan_dg_floor_0p1.py",
    "task3/notebook_blocks/13_train_dan_dg_floor_10.py",
    "task3/preregistration/DAN_DG_BANDWIDTH_FLOOR_STUDY.md",
    "task3/provenance/RUN_LOG.md",
    "task3/research_variants/__init__.py",
    "task3/research_variants/bandwidth_floor.py",
    "task3/research_variants/config.py",
    "task3/research_variants/notebook_runner.py",
    "task3/tests/test_task3_bandwidth_floor.py",
}

head = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"],
    text=True,
).strip()

changed_files = set(
    subprocess.check_output(
        [
            "git",
            "-C",
            str(REPO),
            "diff",
            "--name-only",
            DIAGNOSTIC_COMMIT,
            "HEAD",
        ],
        text=True,
    ).splitlines()
)

missing = EXPECTED_FILES - changed_files
unexpected = changed_files - EXPECTED_FILES
wrong_folder = REPO / "task3 "

print("Repository HEAD:", head)
print("Approved changed files:", len(changed_files))
print("Missing:", sorted(missing))
print("Unexpected:", sorted(unexpected))
print("Trailing-space folder exists:", wrong_folder.exists())

if missing:
    raise RuntimeError(f"Required files are missing: {sorted(missing)}")
if unexpected:
    raise RuntimeError(f"Unexpected changed files found: {sorted(unexpected)}")
if wrong_folder.exists():
    raise RuntimeError("The mistaken trailing-space 'task3 ' folder still exists.")

block10 = (
    REPO
    / "task3"
    / "notebook_blocks"
    / "10_prepare_bandwidth_floor_research_variant.py"
)

if not block10.is_file():
    raise FileNotFoundError(block10)

print("Running:", block10)
runpy.run_path(str(block10), run_name="__main__")

CalledProcessError: Command '['git', '-C', '/content/atml_pa1_task3_source', 'diff', '--name-only', '10e3c1e80a267ffe77456814071fa7895ac2743d', 'HEAD']' returned non-zero exit status 128.

In [ ]:
from pathlib import Path
import runpy
import subprocess

REPO = Path("/content/atml_pa1_task3_source")
DIAGNOSTIC_COMMIT = "10e3c1e80a267ffe77456814071fa7895ac2743d"

is_shallow = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "--is-shallow-repository"],
    text=True,
).strip()

if is_shallow == "true":
    print("Restoring complete Git history...")
    subprocess.run(
        ["git", "-C", str(REPO), "fetch", "--unshallow", "origin"],
        check=True,
    )
else:
    subprocess.run(
        ["git", "-C", str(REPO), "fetch", "origin"],
        check=True,
    )

# Confirm that the required earlier commit is now available.
subprocess.run(
    [
        "git",
        "-C",
        str(REPO),
        "cat-file",
        "-e",
        f"{DIAGNOSTIC_COMMIT}^{{commit}}",
    ],
    check=True,
)

head = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"],
    text=True,
).strip()

print("Repository HEAD:", head)
print("Diagnostic commit available:", DIAGNOSTIC_COMMIT)

block10 = (
    REPO
    / "task3"
    / "notebook_blocks"
    / "10_prepare_bandwidth_floor_research_variant.py"
)

if not block10.is_file():
    raise FileNotFoundError(block10)

print("Running:", block10)
runpy.run_path(str(block10), run_name="__main__")

Restoring complete Git history...
Repository HEAD: dc3acfdf547e7bc29bd381b3fe05e271879f18d0
Diagnostic commit available: 10e3c1e80a267ffe77456814071fa7895ac2743d
Running: /content/atml_pa1_task3_source/task3/notebook_blocks/10_prepare_bandwidth_floor_research_variant.py
All Block 10 pre-calibration gates passed.
Repository commit: dc3acfdf547e7bc29bd381b3fe05e271879f18d0
Target-free tests passed: 25/25
Calibration: common initialization, 235 source-only center-crop batches
Sketch images accessible to calibration: 0

Audited Block 10 completion:
 {
  "status": "TASK3_BLOCK_10_BANDWIDTH_FLOOR_CALIBRATION_PASS",
  "variant_id": "dan_dg_initial_bandwidth_floor_v1",
  "variant_protocol_version": "task3-research-bandwidth-floor-2026-09-25-v1",
  "research_variant": true,
  "primary_protocol_replacement": false,
  "source_only_phase": true,
  "calibration_completed": true,
  "training_started": false,
  "final_evaluation_started": false,
  "sketch_images_accessed": 0,
  "repository_commit": "

{'__name__': '__main__',
 '__doc__': 'Block 10: install, verify, authorize, and calibrate the research variant.\n\nRun only after committing the complete approved research-variant file set. This block\ndoes not train a model. It authenticates the original source-only evidence, verifies\nthe exact Git change set and all 25 target-free tests, writes the explicit authorization\nrecord, and derives the three frozen initialization bandwidth floors without Sketch.\n',
 '__package__': '',
 '__loader__': None,
 '__spec__': None,
 '__file__': '/content/atml_pa1_task3_source/task3/notebook_blocks/10_prepare_bandwidth_floor_research_variant.py',
 '__cached__': None,
 '__builtins__': {'__name__': 'builtins',
  '__doc__': "Built-in functions, types, exceptions, and other objects.\n\nThis module provides direct access to all 'built-in'\nidentifiers of Python; for example, builtins.len is\nthe full name for the built-in function len().\n\nThis module is not normally accessed explicitly by most\nappli

In [ ]:
from pathlib import Path
import runpy

REPO = Path("/content/atml_pa1_task3_source")
block11 = (
    REPO
    / "task3"
    / "notebook_blocks"
    / "11_train_dan_dg_floor_1.py"
)

if not block11.is_file():
    raise FileNotFoundError(block11)

print("Running:", block11)

# Assignment prevents Colab from displaying runpy's internal namespace.
_ = runpy.run_path(str(block11), run_name="__main__")

Running: /content/atml_pa1_task3_source/task3/notebook_blocks/11_train_dan_dg_floor_1.py
All Block 11 pre-training gates passed.
Research variant: dan_dg_initial_bandwidth_floor_v1
Starting: dan_dg_floor_1 lambda_DG = 1.0
Frozen pair floors: {
  "photo__art_painting": 0.8772861361503601,
  "photo__cartoon": 0.8917758464813232,
  "art_painting__cartoon": 0.8030382394790649
}
Only the effective bandwidth median is changed from the primary run.
Sketch images accessible to training: 0

Epoch summary:
epoch=01 classification=0.512677 mmd=0.062144 gradient=24.192760 clip_fraction=0.6213 floor_active=0.9986 mean_f1=0.890451 worst_f1=0.852321
epoch=02 classification=0.233285 mmd=0.024631 gradient=21.955681 clip_fraction=0.5106 floor_active=1.0000 mean_f1=0.892793 worst_f1=0.848136
epoch=03 classification=0.174299 mmd=0.018925 gradient=19.287590 clip_fraction=0.4128 floor_active=1.0000 mean_f1=0.915892 worst_f1=0.891954
epoch=04 classification=0.122131 mmd=0.016491 gradient=16.675914 clip_fract

In [ ]:
from pathlib import Path
import runpy

REPO = Path("/content/atml_pa1_task3_source")
block12 = (
    REPO
    / "task3"
    / "notebook_blocks"
    / "12_train_dan_dg_floor_0p1.py"
)

if not block12.is_file():
    raise FileNotFoundError(block12)

print("Running:", block12)
_ = runpy.run_path(str(block12), run_name="__main__")

Running: /content/atml_pa1_task3_source/task3/notebook_blocks/12_train_dan_dg_floor_0p1.py
All Block 12 pre-training gates passed.
Research variant: dan_dg_initial_bandwidth_floor_v1
Starting: dan_dg_floor_0p1 lambda_DG = 0.1
Frozen pair floors: {
  "photo__art_painting": 0.8772861361503601,
  "photo__cartoon": 0.8917758464813232,
  "art_painting__cartoon": 0.8030382394790649
}
Only the effective bandwidth median is changed from the primary run.
Sketch images accessible to training: 0

Epoch summary:
epoch=01 classification=0.469674 mmd=0.332626 gradient=11.730703 clip_fraction=0.0511 floor_active=0.9901 mean_f1=0.884994 worst_f1=0.841470
epoch=02 classification=0.208458 mmd=0.272135 gradient=9.362289 clip_fraction=0.0255 floor_active=1.0000 mean_f1=0.916294 worst_f1=0.893130
epoch=03 classification=0.134645 mmd=0.226686 gradient=8.052371 clip_fraction=0.0213 floor_active=1.0000 mean_f1=0.897283 worst_f1=0.855438
epoch=04 classification=0.119947 mmd=0.217604 gradient=7.455461 clip_frac

In [ ]:
from pathlib import Path
import runpy

REPO = Path("/content/atml_pa1_task3_source")
block13 = (
    REPO
    / "task3"
    / "notebook_blocks"
    / "13_train_dan_dg_floor_10.py"
)

if not block13.is_file():
    raise FileNotFoundError(block13)

print("Running:", block13)
_ = runpy.run_path(str(block13), run_name="__main__")

Running: /content/atml_pa1_task3_source/task3/notebook_blocks/13_train_dan_dg_floor_10.py
All Block 13 pre-training gates passed.
Research variant: dan_dg_initial_bandwidth_floor_v1
Starting: dan_dg_floor_10 lambda_DG = 10.0
Frozen pair floors: {
  "photo__art_painting": 0.8772861361503601,
  "photo__cartoon": 0.8917758464813232,
  "art_painting__cartoon": 0.8030382394790649
}
Only the effective bandwidth median is changed from the primary run.
Sketch images accessible to training: 0

Epoch summary:
epoch=01 classification=1.338495 mmd=0.015560 gradient=52.777291 clip_fraction=0.9234 floor_active=0.9986 mean_f1=0.520139 worst_f1=0.458576
epoch=02 classification=0.659959 mmd=0.006574 gradient=61.084234 clip_fraction=0.9872 floor_active=1.0000 mean_f1=0.856461 worst_f1=0.830745
epoch=03 classification=0.430851 mmd=0.004817 gradient=52.635111 clip_fraction=0.9702 floor_active=1.0000 mean_f1=0.860768 worst_f1=0.823911
epoch=04 classification=0.328543 mmd=0.004226 gradient=49.890172 clip_fr

In [10]:
from pathlib import Path
import runpy
import subprocess

REPO = Path("/content/atml_pa1_task3_source")

subprocess.run(
    ["git", "-C", str(REPO), "pull", "--ff-only"],
    check=True,
)

head = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"],
    text=True,
).strip()

block14 = (
    REPO
    / "task3"
    / "notebook_blocks"
    / "14_run_bandwidth_floor_source_diagnostics.py"
)

print("Repository HEAD:", head)
print("Running:", block14)

if not block14.is_file():
    raise FileNotFoundError(block14)

_ = runpy.run_path(str(block14), run_name="__main__")

Repository HEAD: 136e28b556d38c99e486e2d53ddff33d921fbfc6
Running: /content/atml_pa1_task3_source/task3/notebook_blocks/14_run_bandwidth_floor_source_diagnostics.py
All Block 14 pre-diagnostic gates passed.
Training commit preserved: dc3acfdf547e7bc29bd381b3fe05e271879f18d0
Diagnostic commit: 136e28b556d38c99e486e2d53ddff33d921fbfc6
Target-free tests passed: 29/29
Previously completed and authenticated: ERM, original lambda 1, SAM
New diagnostics: original lambda 0.1 and 10; stabilized lambda 1, 0.1, and 10
Unified final coverage: all eight completed checkpoints
Probe and sharpness subsets: exact Block 09 designs reused
Sketch images accessible to diagnostics: 0


Audited all-model source-diagnostic summary:

{
  "status": "TASK3_BLOCK_14_ALL_SOURCE_DIAGNOSTICS_COVERAGE_PASS",
  "variant_id": "dan_dg_initial_bandwidth_floor_v1",
  "variant_protocol_version": "task3-research-bandwidth-floor-2026-09-25-v1",
  "research_variant": true,
  "primary_protocol_replacement": false,
  "source_on

In [11]:
import runpy
import subprocess
from pathlib import Path

CODE_ROOT = Path("/content/atml_pa1_task3_source")

subprocess.run(
    ["git", "pull", "--ff-only"],
    cwd=CODE_ROOT,
    check=True,
)

block15 = CODE_ROOT / "task3/notebook_blocks/15_create_final_experiment_lock.py"

print("Repository HEAD:", subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=CODE_ROOT,
    text=True,
).strip())
print("Running:", block15)

runpy.run_path(str(block15), run_name="__main__")

Repository HEAD: 31618caebaf42acb18dd657407d22f03b4a2464f
Running: /content/atml_pa1_task3_source/task3/notebook_blocks/15_create_final_experiment_lock.py
All Block 15 target-free gates passed.
All eight selected checkpoints are now immutable.
Sketch records parsed: 0
Sketch images accessed: 0
Task 2 target-result files read: 0
Final evaluation started: False

Audited final-lock completion:

{
  "status": "TASK3_BLOCK_15_FINAL_EXPERIMENT_LOCK_PASS",
  "target_free_lock_creation": true,
  "target_labels_accessed": false,
  "sketch_images_accessed": 0,
  "final_evaluation_started": false,
  "repository_commit": "31618caebaf42acb18dd657407d22f03b4a2464f",
  "code_tree_sha256": "e5d2bf6341c341d532aeccbae89a34dca01ff3b7dc2e8dfddc782d04b58f5b05",
  "target_free_tests_passed": true,
  "final_lock": {
    "path": "/content/drive/MyDrive/ATML-PA1/task3_domain_generalization_20260924/provenance/final_experiment_lock.json",
    "sha256": "35d2a2df7e2b64ac14eb05c260b8b24410b5485aafa489d9427368b3ca

{'__name__': '__main__',
 '__doc__': 'Block 15: authenticate all source-only evidence and create the final lock.\n\nThis block deliberately does not parse target records, list archive members, open\nSketch images, or read Task 2 target-bearing result files.\n',
 '__package__': '',
 '__loader__': None,
 '__spec__': None,
 '__file__': '/content/atml_pa1_task3_source/task3/notebook_blocks/15_create_final_experiment_lock.py',
 '__cached__': None,
 '__builtins__': {'__name__': 'builtins',
  '__doc__': "Built-in functions, types, exceptions, and other objects.\n\nThis module provides direct access to all 'built-in'\nidentifiers of Python; for example, builtins.len is\nthe full name for the built-in function len().\n\nThis module is not normally accessed explicitly by most\napplications, but can be useful in modules that provide\nobjects with the same name as a built-in value, but in\nwhich the built-in of that name is also needed.",
  '__package__': '',
  '__loader__': _frozen_importlib.Buil

In [12]:
import os
import runpy
import subprocess
from pathlib import Path

CODE_ROOT = Path("/content/atml_pa1_task3_source")

os.environ["TASK3_APPROVED_FINAL_LOCK_SHA256"] = (
    "35d2a2df7e2b64ac14eb05c260b8b24410b5485aafa489d9427368b3caa52348"
)

block16 = CODE_ROOT / "task3/notebook_blocks/16_run_final_sketch_evaluation.py"

print("Repository HEAD:", subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=CODE_ROOT,
    text=True,
).strip())
print("Running:", block16)

_block16_namespace = runpy.run_path(str(block16), run_name="__main__")
del _block16_namespace

Repository HEAD: 31618caebaf42acb18dd657407d22f03b4a2464f
Running: /content/atml_pa1_task3_source/task3/notebook_blocks/16_run_final_sketch_evaluation.py
All Block 16 post-lock gates passed.
Final lock: 35d2a2df7e2b64ac14eb05c260b8b24410b5485aafa489d9427368b3caa52348
All eight checkpoints will be evaluated exactly once on 3,929 Sketch images.
No target result can change training, selection, or checkpoint identity.

Final evaluation completed. No checkpoint was changed.

{
  "status": "TASK3_BLOCK_16_FINAL_SKETCH_EVALUATION_PASS",
  "experiment_lock_sha256": "35d2a2df7e2b64ac14eb05c260b8b24410b5485aafa489d9427368b3caa52348",
  "repository_commit": "31618caebaf42acb18dd657407d22f03b4a2464f",
  "code_tree_sha256": "e5d2bf6341c341d532aeccbae89a34dca01ff3b7dc2e8dfddc782d04b58f5b05",
  "target_count": 3929,
  "sketch_images_accessed": 3929,
  "target_labels_used_for_training_or_selection": false,
  "all_eight_checkpoints_evaluated": true,
  "main_comparison": [
    "erm",
    "dan_dg_1",
   